# HMM-Driven Mixture of Experts for Financial Forecasting
## Implementation and Code Reference

This notebook provides an organized implementation reference for the main methodological and experimental components developed in the study *Disentangling Regime Segmentation, Expert Specialization, and Routing in HMM-Driven Mixture of Experts for Financial Forecasting* (A. Lazcano, J. E. Sandubete, L. Calderón). It brings together the code for synthetic time-series generation, data preprocessing, HMM-based regime identification, neural forecasting architectures, Hard-Routing and Soft-Routing Mixture of Experts models, forecasting benchmarks, evaluation procedures, and supplementary analyses.

The code is organized into thematic sections to facilitate inspection, understanding, and reuse of the different components of the experimental framework.

# 1. Notebook Overview

## 1.1 Scope

The study proposes a regime-aware hybrid forecasting architecture in which a Hidden Markov Model, fitted exclusively on the training partition to prevent information leakage, identifies latent market regimes, and a shared-backbone Transformer with regime-specific expert heads produces one-step-ahead forecasts through a soft, probability-weighted combination of expert outputs. This notebook documents the main methodological components of that implementation:

- Chronological split of each series: first 80% of the observations for model estimation, hyperparameter selection and training, and last 20% for out-of-sample evaluation.
- One-step-ahead forecasting from rolling windows of 30 observations.
- HMM with Gaussian emissions and full covariance, estimated on standardised log-returns and 5-day rolling volatility. The number of states (2 to 7) is selected with the Bayesian Information Criterion on the training partition only, and the regimes of the whole series are obtained by forward filtering.
- Input features of the MoE models: min–max normalised log-returns, 5-day rolling volatility, 14-day RSI and one-hot encoded HMM states.
- Evaluation on the reconstructed price scale (MSE, MAE, RMSE, R²) and on the return scale (RMSE, R²). Predicted prices are anchored on the last observed price.

## 1.2 How to read this notebook

The blocks of code fall into three categories:

- **Core implementation** (Sections 2–10): functions, architectures and procedures that implement the main methodological components.
- **Execution-specific configuration**: cells whose settings (seeds, datasets, file names) reflect one of the executions used during development. They are preceded by an explicit note and are preserved without modification.
- **Supplementary / development implementations** (Section 11): additional analyses and implementations retained for traceability. They are self-contained and are not a required part of the main pipeline.

| Section | Content | Type |
| --- | --- | --- |
| 1 | Overview, installation and global configuration | Overview / execution-specific configuration |
| 2 | Synthetic time-series generation | Core implementation (self-contained block) |
| 3 | Data loading and preprocessing | Core implementation |
| 4 | HMM-based regime identification | Core implementation |
| 5 | Neural architectures | Core implementation |
| 6 | Hard-Routing HMM-MoE | Core implementation |
| 7 | Soft-Routing Mixture of Experts | Core implementation |
| 8 | Global Transformer benchmark | Core implementation (integrated in Section 7) |
| 9 | Evaluation metrics, statistical functions and main execution script | Core implementation / execution-specific configuration |
| 10 | Random Walk benchmark | Core implementation (self-contained block) |
| 11 | Additional multi-seed and statistical analysis | Supplementary / additional analysis (self-contained block) |

The cells of Sections 1 and 3–9 share the same Python namespace, whereas Sections 2, 10 and 11 declare their own dependencies, imports and configuration. The main pipeline reads daily price files in CSV format with a date column (`Date`) and a price column (`Price`, `Close` or `Adj Close`). Intermediate and final results (HMM states, per-seed metrics, aggregated metrics, Diebold–Mariano results) are written as CSV files to the working directory.

## 1.3 Environment and global configuration

Installation of the HMM library, imports, seed-setting utility and global configuration of the main pipeline: seeds, window length, train/test ratio, number of epochs, batch size, HMM settings and datasets.

> **Execution configuration.**
> The following block preserves the configuration used for this experimental execution.

The cell below has a single seed and a single dataset active, while the remaining options are kept commented.

### 1.3.1 Installation

In [ ]:
!pip install hmmlearn --quiet

### 1.3.2 Imports and global configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict
from math import sqrt

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Bidirectional, Flatten,
    MultiHeadAttention, Dropout, LayerNormalization, GlobalAveragePooling1D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import ParameterGrid

from hmmlearn.hmm import GaussianHMM
from scipy.stats import multivariate_normal
from scipy.special import logsumexp

warnings.filterwarnings("ignore")

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    np.random.seed(seed)
    import random
    random.seed(seed)
    tf.random.set_seed(seed)

#SEEDS = [42, 123, 2024]
SEEDS = [42]

CACHE_GRID_SEARCH = True
SEED_GRID = 42
GRID_FAST = False

SEQ_LENGTH  = 30
TRAIN_RATIO = 0.80
EPOCHS      = 60
BATCH_SIZE  = 32

HMM_MIN_STATES = 2
HMM_MAX_STATES = 7
HMM_VOL_WINDOW = 5
HMM_COV_TYPE   = "full"

DATASETS = [
    #("APPLE.csv",  "APPLE"),
    #("NVIDIA.csv", "NVIDIA"),
    ("SP500v3.csv",  "SSP500")#,
]

# 2. Synthetic Time-Series Generation

Code for the generation of synthetic price series with a controlled latent-state structure. The block is self-contained: it declares its own dependencies, imports and configuration. It implements the following components:

- **Latent state generation.** The hidden-state sequence is a first-order Markov chain sampled directly from the transition matrix (`generar_cadena_markov`), without external HMM libraries.
- **Transition matrix.** Symmetric structure: a common self-transition probability (`_P_DIAG`) on the diagonal and the remaining probability mass distributed uniformly among the other states, which yields an expected regime duration of 1 / (1 − `_P_DIAG`) observations. The initial-state distribution is uniform.
- **Gaussian emissions.** Conditional on the active regime, each log-return is drawn from a state-specific Gaussian distribution defined by a mean and a standard deviation (`generar_log_returns`).
- **Price reconstruction.** Prices are obtained from the cumulative log-returns, starting from a fixed initial price (`PRECIO_INICIAL`), and are assigned business-day dates.
- **State-representation criterion in the test block.** Each series is split chronologically (80% train, 20% test). The series is regenerated with a new random seed (`seed_inicial + attempt`) until all latent states appear in the test block and each of them accounts for at least a minimum share of its observations (`MIN_FRACCION_ESTADO_TEST`).
- **Outputs.** Series (CSV), segment tables (consecutive runs of the same state), figures with the train/test cut, a global summary and a ZIP file.

## 2.0 Dependencies and imports

In [ ]:
# ============================================================
# GENERADOR DE SERIES SINTETICAS HMM - 4 ESTADOS OCULTOS
# ============================================================

!pip install numpy pandas matplotlib openpyxl

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2.1 Global parameters (states, emission parameters, transition matrix, sizes and seeds)

> **Execution-specific configuration.**
> The series sizes (`TAMANOS`), the base seed (`SEMILLA_BASE`) and the numerical parameters below reflect one of the experimental executions used during development and are preserved without modification for traceability.

> **Usage note – six-state series.** The code is configured for a four-state structure (STS-4). A six-state series (STS-6) is generated with exactly the same code: it is only necessary to enter the corresponding values in the parameter cell below, namely `N_STATES`, the entries of `STATE_LABELS`, `MEANS`, `STDS` and `PALETA` (one per state) and the diagonal (self-transition) probability `_P_DIAG`, using the values reported in Appendix A.1 of the manuscript. Output names such as `SERIE_4_ESTADOS_N...` and `CARPETA_SALIDA` are only labels and can be renamed accordingly.

In [ ]:
# ------------------------------------------------------------
# 1) PARAMETROS GLOBALES
# ------------------------------------------------------------

N_STATES = 4

# Etiquetas descriptivas de cada regimen
STATE_LABELS = {
    0: "Estable (baja vol.)",
    1: "Alcista (vol. moderada)",
    2: "Bajista (vol. mod/alta)",
    3: "Turbulento (alta vol.)",
}

# Media y desviacion tipica del log-retorno por estado
MEANS = np.array([0.0002, 0.0010, -0.0010, 0.0000])
STDS  = np.array([0.005,  0.008,   0.010,   0.020])

# Matriz de transicion: alta persistencia en la diagonal ->
# duracion media de cada estado = 1 / (1 - p_diagonal) ~ 250 dias (~1 anio bursatil)
# (subido de nuevo: 0.94 -> 0.985 -> 0.996, para tramos mucho mas largos)
_P_DIAG = 0.996
_P_OFF = (1.0 - _P_DIAG) / (N_STATES - 1)
TRANSMAT = np.full((N_STATES, N_STATES), _P_OFF)
np.fill_diagonal(TRANSMAT, _P_DIAG)

# Probabilidad inicial uniforme (sin sesgo de arranque)
STARTPROB = np.full(N_STATES, 1.0 / N_STATES)

# Tamanos de serie a generar (todas >= 5000 obs.)
TAMANOS = [5000, 7500, 10000, 12500, 15000]

PRECIO_INICIAL = 100.0
FECHA_INICIO   = "2010-01-01"
TRAIN_RATIO    = 0.80

SEMILLA_BASE   = 1000        # semilla de partida (se incrementa si hay que regenerar)
MAX_INTENTOS   = 2000        # limite de reintentos de regeneracion por serie

# No basta con que cada estado "aparezca" en el test (podria ser solo 2-3
# observaciones sueltas). Exigimos que cada estado represente al menos este
# porcentaje del bloque de test, para que la distribucion sea realmente
# equilibrada y no un vestigio anecdotico.
MIN_FRACCION_ESTADO_TEST = 0.10   # cada estado >= 10% de las observaciones de test

CARPETA_SALIDA = "series_sinteticas_hmm_4_estados"
os.makedirs(CARPETA_SALIDA, exist_ok=True)

PALETA = ["#27ae60", "#2980b9", "#e74c3c", "#8e44ad"]  # un color por estado

## 2.2 Markov-chain generation and log-return sampling

In [ ]:
# ------------------------------------------------------------
# 2) GENERACION CONTROLADA DE LA CADENA DE MARKOV
# ------------------------------------------------------------
def generar_cadena_markov(n_samples: int, transmat: np.ndarray,
                           startprob: np.ndarray, seed: int) -> np.ndarray:
    """
    Genera una secuencia de estados ocultos muestreando directamente
    la matriz de transicion (sin librerias de HMM externas).
    Esto da control total sobre la cadena y permite regenerarla con
    una nueva semilla si no cumple la condicion de test.
    """
    rng = np.random.default_rng(seed)
    n_states = transmat.shape[0]
    states = np.empty(n_samples, dtype=int)
    states[0] = rng.choice(n_states, p=startprob)
    for t in range(1, n_samples):
        states[t] = rng.choice(n_states, p=transmat[states[t - 1]])
    return states, rng


def generar_log_returns(states: np.ndarray, means: np.ndarray,
                         stds: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Muestrea el log-retorno de cada observacion segun su estado oculto."""
    return rng.normal(loc=means[states], scale=stds[states])

## 2.3 Test-block validation (presence and minimum share of every state)

In [ ]:
# ------------------------------------------------------------
# 3) VALIDACION: LOS 4 ESTADOS DEBEN APARECER EN EL TEST
# ------------------------------------------------------------
def cumple_condicion_test(states: np.ndarray, train_ratio: float, n_states: int,
                           min_fraccion: float = MIN_FRACCION_ESTADO_TEST) -> bool:
    """
    Comprueba que el bloque final (test = ultimo (1-train_ratio) %)
    contiene los n_states estados Y que ademas estan bien distribuidos:
    cada estado debe representar al menos `min_fraccion` de las
    observaciones de test (no solo aparecer una vez de forma anecdotica).
    """
    n = len(states)
    corte = int(n * train_ratio)
    estados_test = states[corte:]

    # 1) presencia de los n_states estados
    if set(np.unique(estados_test)) != set(range(n_states)):
        return False

    # 2) distribucion minima por estado
    conteos = np.bincount(estados_test, minlength=n_states)
    proporciones = conteos / len(estados_test)
    return bool(np.all(proporciones >= min_fraccion))


def generar_serie_valida(n_samples: int, transmat: np.ndarray, startprob: np.ndarray,
                          means: np.ndarray, stds: np.ndarray, train_ratio: float,
                          n_states: int, seed_inicial: int, max_intentos: int,
                          min_fraccion: float = MIN_FRACCION_ESTADO_TEST):
    """
    Genera estados + log-retornos y REGENERA con una nueva semilla
    (seed_inicial + intento) hasta que el bloque de test contenga
    los n_states estados CON una distribucion minima aceptable cada uno
    (>= min_fraccion). Este bucle es la garantia mecanica pedida en el
    encargo para asegurar un test bien distribuido, no solo "completo".
    """
    for intento in range(max_intentos):
        seed = seed_inicial + intento
        states, rng = generar_cadena_markov(n_samples, transmat, startprob, seed)
        if cumple_condicion_test(states, train_ratio, n_states, min_fraccion):
            log_returns = generar_log_returns(states, means, stds, rng)
            return states, log_returns, seed, intento + 1
    raise RuntimeError(
        f"No se logro una serie valida en {max_intentos} intentos para n={n_samples}. "
        "Prueba a aumentar max_intentos, bajar min_fraccion, o reducir la "
        "persistencia de la diagonal (TRANSMAT)."
    )

## 2.4 Segment table (consecutive runs of the same hidden state)

In [ ]:
# ------------------------------------------------------------
# 4) TABLA DE SEGMENTOS (tramos consecutivos del mismo estado)
# ------------------------------------------------------------
def construir_tabla_segmentos(df: pd.DataFrame) -> pd.DataFrame:
    """Resume la serie en tramos consecutivos del mismo Hidden_State."""
    states = df["Hidden_State"].values
    dates = df["Date"].values
    splits = df["Split"].values

    filas = []
    seg_id = 1
    inicio = 0
    n = len(states)

    for t in range(1, n + 1):
        if t == n or states[t] != states[inicio]:
            fin = t - 1
            filas.append({
                "Segment_ID":   seg_id,
                "Start_Index":  inicio + 1,          # 1-based, como en el ejemplo del encargo
                "End_Index":    fin + 1,
                "Start_Date":   dates[inicio],
                "End_Date":     dates[fin],
                "Hidden_State": states[inicio],
                "Length":       fin - inicio + 1,
                "Split_Start":  splits[inicio],
                "Split_End":    splits[fin],
            })
            seg_id += 1
            inicio = t

    return pd.DataFrame(filas)

## 2.5 Figure: price coloured by hidden state with the train/test cut

In [ ]:
# ------------------------------------------------------------
# 5) GRAFICO: PRECIO COLOREADO POR ESTADO + CORTE TRAIN/TEST
# ------------------------------------------------------------
def graficar_serie(df: pd.DataFrame, nombre_serie: str, corte_idx: int, ruta_png: str):
    fig, ax = plt.subplots(figsize=(15, 5.5))

    # Linea base fina de fondo para ver la trayectoria continua
    ax.plot(df["Date"], df["Price"], color="#bdc3c7", lw=0.6, alpha=0.8, zorder=1)

    # Puntos coloreados por estado oculto real
    for estado in range(N_STATES):
        m = df["Hidden_State"] == estado
        ax.scatter(df.loc[m, "Date"], df.loc[m, "Price"],
                   s=6, color=PALETA[estado], zorder=2,
                   label=f"S{estado}: {STATE_LABELS[estado]}")

    # Linea vertical en el corte train/test
    fecha_corte = df["Date"].iloc[corte_idx]
    ax.axvline(fecha_corte, color="black", linestyle="--", lw=1.4, zorder=3)

    # Anotaciones "Train 80%" / "Test 20%"
    y_top = df["Price"].max()
    ax.annotate("Train 80%", xy=(df["Date"].iloc[corte_idx // 2], y_top),
                ha="center", va="bottom", fontsize=10, fontweight="bold", color="#2c3e50")
    idx_test_mid = corte_idx + (len(df) - corte_idx) // 2
    ax.annotate("Test 20%", xy=(df["Date"].iloc[idx_test_mid], y_top),
                ha="center", va="bottom", fontsize=10, fontweight="bold", color="#c0392b")

    ax.set_title(f"{nombre_serie} — N = {len(df):,} observaciones",
                fontsize=13, fontweight="bold")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Precio")
    ax.legend(markerscale=3, fontsize=9, ncol=N_STATES, loc="lower center",
             bbox_to_anchor=(0.5, -0.28))
    ax.grid(True, alpha=0.25)

    plt.tight_layout()
    plt.savefig(ruta_png, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 2.6 Generation pipeline (series, CSV files, segment tables, figures and global summary)

In [ ]:
# ------------------------------------------------------------
# 6) PIPELINE PRINCIPAL: GENERAR TODAS LAS SERIES
# ------------------------------------------------------------
def generar_todas_las_series():
    resumen_global_filas = []

    for n_samples in TAMANOS:
        nombre_serie = f"SERIE_4_ESTADOS_N{n_samples}"
        print("\n" + "=" * 80)
        print(nombre_serie)
        print("=" * 80)

        # --- generar estados + retornos validos (con reintentos si hace falta) ---
        states, log_returns, seed_usada, n_intentos = generar_serie_valida(
            n_samples=n_samples, transmat=TRANSMAT, startprob=STARTPROB,
            means=MEANS, stds=STDS, train_ratio=TRAIN_RATIO, n_states=N_STATES,
            seed_inicial=SEMILLA_BASE + n_samples, max_intentos=MAX_INTENTOS
        )

        # --- reconstruir precios: price_t = price_0 * exp(cumsum(log_returns)) ---
        prices = PRECIO_INICIAL * np.exp(np.cumsum(log_returns))
        prices = np.insert(prices, 0, PRECIO_INICIAL)[:n_samples]  # alinear longitudes

        fechas = pd.bdate_range(start=FECHA_INICIO, periods=n_samples)

        corte_idx = int(n_samples * TRAIN_RATIO)
        split = np.array(["Train"] * corte_idx + ["Test"] * (n_samples - corte_idx))

        df = pd.DataFrame({
            "Date":         fechas,
            "Index":        np.arange(1, n_samples + 1),
            "Price":        prices,
            "Log_Return":   log_returns,
            "Hidden_State": states,
            "Split":        split,
        })

        # --- validaciones automaticas exigidas ---
        estados_train = sorted(df.loc[df["Split"] == "Train", "Hidden_State"].unique().tolist())
        estados_test  = sorted(df.loc[df["Split"] == "Test",  "Hidden_State"].unique().tolist())

        # distribucion (%) de cada estado dentro del bloque de test
        test_states_arr = df.loc[df["Split"] == "Test", "Hidden_State"].values
        conteos_test = np.bincount(test_states_arr, minlength=N_STATES)
        pct_test = conteos_test / len(test_states_arr) * 100

        assert len(set(states)) == N_STATES, "La serie completa no contiene los 4 estados."
        assert len(set(estados_test)) == N_STATES, "El test no contiene los 4 estados."
        assert set(estados_test) == set(range(N_STATES)), "El test no cubre exactamente {0,1,2,3}."
        assert np.all(conteos_test / len(test_states_arr) >= MIN_FRACCION_ESTADO_TEST), (
            f"Algun estado no alcanza el {MIN_FRACCION_ESTADO_TEST*100:.0f}% minimo en test: "
            f"{dict(enumerate(np.round(pct_test, 1)))}"
        )

        # --- tabla de segmentos ---
        df_segmentos = construir_tabla_segmentos(df)

        # --- guardar CSV de la serie ---
        ruta_csv = os.path.join(CARPETA_SALIDA, f"{nombre_serie}.csv")
        df.to_csv(ruta_csv, index=False)

        # --- guardar tabla resumen de segmentos ---
        ruta_segmentos = os.path.join(CARPETA_SALIDA, f"RESUMEN_SEGMENTOS_4_ESTADOS_N{n_samples}.csv")
        df_segmentos.to_csv(ruta_segmentos, index=False)

        # --- generar y guardar grafico ---
        ruta_png = os.path.join(CARPETA_SALIDA, f"GRAFICO_ESTADOS_4_ESTADOS_N{n_samples}.png")
        graficar_serie(df, nombre_serie, corte_idx, ruta_png)

        # --- fila para el resumen global ---
        longitudes = df_segmentos["Length"]
        resumen_global_filas.append({
            "Series_Name":            nombre_serie,
            "N_Observations":         n_samples,
            "Train_Size":             corte_idx,
            "Test_Size":              n_samples - corte_idx,
            "States_In_Train":        str(estados_train),
            "States_In_Test":         str(estados_test),
            "All_States_In_Test":     set(estados_test) == set(range(N_STATES)),
            "Test_Pct_Estado_0":      round(pct_test[0], 2),
            "Test_Pct_Estado_1":      round(pct_test[1], 2),
            "Test_Pct_Estado_2":      round(pct_test[2], 2),
            "Test_Pct_Estado_3":      round(pct_test[3], 2),
            "Test_Bien_Distribuido":  bool(np.all(pct_test / 100 >= MIN_FRACCION_ESTADO_TEST)),
            "Number_Of_Segments":     len(df_segmentos),
            "Average_Segment_Length": round(longitudes.mean(), 2),
            "Min_Segment_Length":     int(longitudes.min()),
            "Max_Segment_Length":     int(longitudes.max()),
            "Seed_Used":              seed_usada,
            "Attempts_Needed":        n_intentos,
        })

        # --- resumen por pantalla exigido en el punto 8 ---
        print(f"Serie N={n_samples} generada correctamente.")
        print(f"  Semilla usada: {seed_usada}  (intentos de regeneracion: {n_intentos})")
        print(f"  Estados en train: {estados_train}")
        print(f"  Estados en test : {estados_test}")
        print(f"  Distribucion en test: " +
              ", ".join(f"S{i}={pct_test[i]:.1f}%" for i in range(N_STATES)) +
              f"  (minimo exigido: {MIN_FRACCION_ESTADO_TEST*100:.0f}% cada uno)")
        print(f"  Numero de segmentos: {len(df_segmentos)}")
        print(f"  Longitud media de segmentos: {longitudes.mean():.1f}")
        print(f"  Archivos guardados: {ruta_csv}, {ruta_segmentos}, {ruta_png}")

    # --- guardar resumen global de todas las series ---
    df_resumen_global = pd.DataFrame(resumen_global_filas)
    ruta_resumen_global = os.path.join(CARPETA_SALIDA, "RESUMEN_GLOBAL_SERIES_4_ESTADOS.csv")
    df_resumen_global.to_csv(ruta_resumen_global, index=False)

    print("\n" + "=" * 80)
    print("RESUMEN GLOBAL")
    print("=" * 80)
    print(df_resumen_global.to_string(index=False))
    print(f"\nGuardado: {ruta_resumen_global}")

    return df_resumen_global

## 2.7 Packaging of the results in a ZIP file

In [ ]:
# ------------------------------------------------------------
# 7) EMPAQUETAR TODO EN UN ZIP (util para descargar de Colab)
# ------------------------------------------------------------
def empaquetar_resultados(zip_name: str = "series_sinteticas_hmm_4_estados.zip"):
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        for archivo in os.listdir(CARPETA_SALIDA):
            ruta = os.path.join(CARPETA_SALIDA, archivo)
            zf.write(ruta, arcname=archivo)
    print(f"\nZIP creado: {zip_name}")

    try:
        from google.colab import files
        files.download(zip_name)
    except ImportError:
        print("(descarga automatica solo disponible en Google Colab; "
              "descarga el zip manualmente desde el explorador de archivos)")

## 2.8 Execution

In [ ]:
# ------------------------------------------------------------
# 8) EJECUCION
# ------------------------------------------------------------
if __name__ == "__main__":
    df_resumen_global = generar_todas_las_series()
    empaquetar_resultados()

# 3. Data Loading and Preprocessing

Functions for loading price series, computing log-returns and rolling volatility, and building the input sequences and technical indicators used by the models. Scaling is applied within the HMM and training functions of the following sections, always with statistics estimated on the training partition: standardisation of log-returns and volatility for the HMM (Section 4), standardisation of log-returns for the hard-routing experts (Section 6), and min–max normalisation of log-returns with scaling of the volatility by its training maximum for the Soft-Routing MoE (Section 7).

## 3.1 Price loading and construction of log-returns and rolling volatility

In [ ]:
def cargar_precio(path_csv, col_fecha="Date", col_precio=None):
    df = pd.read_csv(path_csv, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]
    if col_precio is None:
        candidatos = ["Price", "Close", "Adj Close", "close", "price", "Adj_Close", "adj_close"]
        col_precio = next((c for c in candidatos if c in df.columns), None)
        if col_precio is None:
            raise KeyError(f"No encuentro columna de precio en {path_csv}. Columnas: {list(df.columns)}.")
        print(f"Columna de precio detectada en {path_csv}: '{col_precio}'")
    if col_fecha not in df.columns:
        candidatos_fecha = ["Date", "date", "Fecha", "DATE"]
        col_fecha = next((c for c in candidatos_fecha if c in df.columns), None)
        if col_fecha is None:
            raise KeyError(f"No encuentro columna de fecha en {path_csv}.")
    df[col_fecha] = pd.to_datetime(df[col_fecha])
    df = df.sort_values(col_fecha).reset_index(drop=True)
    df = df[[col_fecha, col_precio]].dropna()
    df = df.rename(columns={col_fecha: "Date", col_precio: "Price"})
    return df

def construir_features_hmm(df, vol_window=HMM_VOL_WINDOW):
    df = df.copy()
    df["Log_Return"] = np.log(df["Price"] / df["Price"].shift(1))
    df["Volatility"] = df["Log_Return"].rolling(vol_window).std()
    df = df.dropna().reset_index(drop=True)
    return df

## 3.2 Sequence construction and RSI

Construction of univariate and multivariate rolling windows (30 observations by default) for one-step-ahead forecasting, and computation of the 14-day Relative Strength Index.

In [ ]:
def create_sequences(arr, seq_len):
    arr = np.asarray(arr, dtype=np.float32)
    n = len(arr) - seq_len
    if n <= 0:
        return np.empty((0, seq_len), dtype=np.float32), np.empty(0, dtype=np.float32)
    shape = (n, seq_len)
    strides = (arr.strides[0], arr.strides[0])
    X = np.lib.stride_tricks.as_strided(arr, shape=shape, strides=strides).copy()
    y = arr[seq_len:].copy()
    return X, y

def create_sequences_multivariate(df_features, target_series, seq_len, pred_len=1):
    data = np.asarray(df_features.values, dtype=np.float32)
    targets = np.asarray(target_series.values, dtype=np.float32)
    n = len(data) - seq_len - pred_len + 1
    if n <= 0:
        return (np.empty((0, seq_len, data.shape[1]), dtype=np.float32), np.empty(0, dtype=np.float32))
    n_feat = data.shape[1]
    shape = (n, seq_len, n_feat)
    strides = (data.strides[0], data.strides[0], data.strides[1])
    X = np.lib.stride_tricks.as_strided(data, shape=shape, strides=strides).copy()
    y = targets[seq_len + pred_len - 1: seq_len + pred_len - 1 + n].copy()
    return X, y

def calculate_rsi(series, window=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

# 4. HMM-Based Regime Identification

Leakage-free identification of latent market regimes. The scaler, the number of states and the parameters of the Gaussian HMM are estimated on the training partition only; the regime of every observation is then obtained by forward filtering, using only the information available up to each time step, without smoothing with future observations. The block includes:

- computation of the log-emissions and forward filter;
- selection of the number of states with BIC (and AIC as an additional diagnostic) on the training partition;
- fitting of the `GaussianHMM` and construction of the state sequence and train/test labels;
- frequency, recurrence and duration analysis of the states;
- plot of the states on the price series.

## 4.1 Log-emissions and forward filter

In [ ]:
def compute_log_emissions(model, X):
    n_states = model.n_components
    log_B = np.column_stack([
        multivariate_normal.logpdf(X, mean=model.means_[k], cov=model.covars_[k])
        for k in range(n_states)
    ])
    return log_B

def forward_filter(model, X):
    """Forward filter vectorizado (bucle interno sobre estados eliminado)."""
    log_B = compute_log_emissions(model, X)
    n_obs, n_states = log_B.shape
    log_A = np.log(model.transmat_ + 1e-300)
    log_pi = np.log(model.startprob_ + 1e-300)
    log_alpha = np.zeros((n_obs, n_states))
    log_alpha[0] = log_pi + log_B[0]
    log_alpha[0] -= logsumexp(log_alpha[0])
    for t in range(1, n_obs):
        log_alpha[t] = logsumexp(log_alpha[t - 1][:, None] + log_A, axis=0) + log_B[t]
        log_alpha[t] -= logsumexp(log_alpha[t])
    return np.argmax(log_alpha, axis=1), np.exp(log_alpha)

## 4.2 Selection of the number of states (BIC / AIC, training partition only)

In [ ]:
def seleccionar_n_estados_train(X_train_scaled, min_states=HMM_MIN_STATES,
                                 max_states=HMM_MAX_STATES, cov_type=HMM_COV_TYPE, random_state=42):
    best_model, best_bic, best_n = None, np.inf, None
    resultados = []
    for n_states in range(min_states, max_states + 1):
        model = GaussianHMM(n_components=n_states, covariance_type=cov_type, n_iter=1000, random_state=random_state)
        model.fit(X_train_scaled)
        ll = model.score(X_train_scaled)
        nf = X_train_scaled.shape[1]
        n_params = n_states ** 2 + 2 * n_states * nf - 1
        bic = -2 * ll + n_params * np.log(len(X_train_scaled))
        aic = -2 * ll + 2 * n_params
        resultados.append((n_states, bic, aic))
        if bic < best_bic:
            best_bic, best_model, best_n = bic, model, n_states
    resultados_df = pd.DataFrame(resultados, columns=["States", "BIC", "AIC"])
    resultados_df.set_index("States").plot(marker="o", figsize=(8, 4))
    plt.title("Selección del número de estados (BIC / AIC) -- solo train")
    plt.ylabel("Valor"); plt.grid(True); plt.tight_layout(); plt.show()
    print(f"Numero de estados seleccionado (BIC, solo train): {best_n}")
    return best_model, resultados_df

## 4.3 HMM estimation without data leakage

In [ ]:
def ajustar_hmm_sin_fuga(df, features=("Log_Return", "Volatility"),
                          train_ratio=TRAIN_RATIO, min_states=HMM_MIN_STATES,
                          max_states=HMM_MAX_STATES, cov_type=HMM_COV_TYPE):
    features = list(features)
    n = len(df)
    split_rows = int(train_ratio * n)
    scaler = StandardScaler()
    scaler.fit(df.loc[:split_rows - 1, features])
    X_all_scaled = scaler.transform(df[features])
    X_train_scaled = X_all_scaled[:split_rows]
    print(f"Buscando estados ocultos (solo train, {split_rows} de {n} obs)...")
    model, bic_df = seleccionar_n_estados_train(X_train_scaled, min_states, max_states, cov_type)
    filtered_states, filtered_probs = forward_filter(model, X_all_scaled)
    df_out = df.copy()
    df_out["State"] = filtered_states
    df_out["Split"] = ["train"] * split_rows + ["test"] * (n - split_rows)
    return df_out, model, scaler, split_rows, bic_df

## 4.4 State frequency and duration analysis

In [ ]:
def analizar_frecuencia_duracion(df, col_estado="State", titulo=""):
    print(f"\nFrecuencia y duracion de estados{f' -- {titulo}' if titulo else ''}:")
    print("-" * 70)
    estados = df[col_estado].values
    total_dias = len(estados)
    duraciones_por_estado = defaultdict(list)
    actual_state, duracion_actual = estados[0], 1
    for i in range(1, len(estados)):
        if estados[i] == actual_state:
            duracion_actual += 1
        else:
            duraciones_por_estado[actual_state].append(duracion_actual)
            actual_state, duracion_actual = estados[i], 1
    duraciones_por_estado[actual_state].append(duracion_actual)
    resumen = []
    for estado in sorted(duraciones_por_estado.keys()):
        runs = duraciones_por_estado[estado]
        total = np.sum(runs)
        resumen.append([estado, total, 100*total/total_dias, len(runs), np.mean(runs), np.max(runs)])
    resumen_df = pd.DataFrame(resumen, columns=["Estado","Frecuencia (dias)","Frecuencia (%)","Concurrencia (episodios)","Duracion media","Duracion max."])
    resumen_df["Frecuencia (%)"] = resumen_df["Frecuencia (%)"].round(2)
    resumen_df["Duracion media"] = resumen_df["Duracion media"].round(2)
    print(resumen_df.to_string(index=False))
    sin_rec = resumen_df.loc[resumen_df["Concurrencia (episodios)"]==1, "Estado"].tolist()
    if sin_rec:
        print(f"\nAVISO: estados {sin_rec} con concurrencia=1.")
    else:
        print("\nOK: todos los estados recurren en mas de una racha.")
    return resumen_df

## 4.5 Plot of the HMM states

In [ ]:
def graficar_estados(df, nombre):
    plt.figure(figsize=(14, 5))
    for s in sorted(df["State"].unique()):
        sub = df[df["State"] == s]
        plt.plot(sub["Date"], sub["Price"], ".", label=f"Estado {s}", markersize=3)
    fecha_test = df.loc[df["Split"]=="test", "Date"].iloc[0]
    plt.axvline(fecha_test, color="black", linestyle="--", alpha=0.6, label="Inicio test")
    plt.title(f"Estados HMM (sin fuga de datos) -- {nombre}")
    plt.xlabel("Fecha"); plt.ylabel("Precio"); plt.legend(); plt.tight_layout(); plt.show()

# 5. Neural Architectures

Definition of the neural components used by the Hard-Routing and Soft-Routing models: positional encoding, MoE fusion layer, Transformer block, individual experts (MLP, LSTM, BiLSTM and Transformer), LSTM gater, global Transformer baseline, regime-specific expert heads and the Soft-Routing MoE Transformer with a shared backbone.

## 5.1 Positional encoding, MoE fusion layer and Transformer block

In [ ]:
class PositionalEncoding(tf.keras.layers.Layer):
    def call(self, x):
        seq_len = tf.shape(x)[1]; d_dim = tf.shape(x)[2]
        positions = tf.cast(tf.range(seq_len)[:, None], tf.float32)
        dims = tf.cast(tf.range(d_dim)[None, :], tf.float32)
        angles = positions / tf.pow(10000.0, (2*(dims//2)) / tf.cast(d_dim, tf.float32))
        sines = tf.math.sin(angles[:, 0::2])
        cosines = tf.math.cos(angles[:, 1::2])
        pe = tf.reshape(tf.stack([sines, cosines], axis=2), [seq_len, -1])[:, :d_dim]
        return x + pe[None, :, :]

In [ ]:
class MoEMergeLayer(tf.keras.layers.Layer):
    def __init__(self, n_experts, **kwargs):
        super().__init__(**kwargs); self.n_experts = n_experts
    def call(self, inputs):
        expert_outputs = inputs[:self.n_experts]
        routing_probs = inputs[-1]
        stacked = tf.stack(expert_outputs, axis=1)
        routed = tf.expand_dims(routing_probs, axis=-1)
        return tf.reduce_sum(stacked * routed, axis=1)

In [ ]:
def transformer_block(x, num_heads, d_model, ff_dim, dropout_rate):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=d_model//num_heads)(x, x)
    attn = Dropout(dropout_rate)(attn)
    x = LayerNormalization(epsilon=1e-6)(x + attn)
    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dropout(dropout_rate)(ffn)
    ffn = Dense(d_model)(ffn)
    ffn = Dropout(dropout_rate)(ffn)
    return LayerNormalization(epsilon=1e-6)(x + ffn)

## 5.2 Individual experts (MLP / LSTM / BiLSTM / Transformer) and model builder from configuration

In [ ]:
def build_expert_model(seq_len, model_type, layers, neurons, activation):
    inp = Input(shape=(seq_len, 1)); x = inp
    if model_type == "LSTM":
        for i in range(layers): x = LSTM(neurons, activation=activation, return_sequences=(i<layers-1))(x)
    elif model_type == "BiLSTM":
        for i in range(layers): x = Bidirectional(LSTM(neurons, activation=activation, return_sequences=(i<layers-1)))(x)
    elif model_type == "MLP":
        x = Flatten()(x)
        for _ in range(layers): x = Dense(neurons, activation=activation)(x)
    return Model(inputs=inp, outputs=Dense(1, activation="linear")(x))

In [ ]:
def build_transformer_expert(seq_len, num_heads, n_blocks, d_model, ff_dim, dropout):
    inp = Input(shape=(seq_len, 1))
    x = Dense(d_model)(inp); x = PositionalEncoding()(x); x = Dropout(dropout)(x)
    for _ in range(n_blocks): x = transformer_block(x, num_heads, d_model, ff_dim, dropout)
    x = GlobalAveragePooling1D()(x); x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x); x = Dense(32, activation="relu")(x)
    return Model(inputs=inp, outputs=Dense(1, activation="linear")(x))

In [ ]:
def build_model_from_cfg(cfg, seq_len):
    if cfg["model_type"] == "Transformer":
        return build_transformer_expert(seq_len, cfg["num_heads"], cfg["n_blocks"], cfg["d_model"], cfg["ff_dim"], cfg["dropout"])
    return build_expert_model(seq_len, cfg["model_type"], cfg["layers"], cfg["neurons"], cfg["activation"])

## 5.3 Gater (LSTM routing network)

LSTM network that produces a probability distribution over the latent regimes through a softmax layer. It is trained in a supervised manner with the HMM state labels.

In [ ]:
def build_gater(seq_len, n_experts, n_features=1):
    inp = Input(shape=(seq_len, n_features))
    x = LSTM(64, activation="tanh")(inp); x = Dense(32, activation="relu")(x)
    return Model(inputs=inp, outputs=Dense(n_experts, activation="softmax")(x))

## 5.4 Global Transformer baseline (architecture)

Transformer encoder without regime-specific heads, used as the global reference model. It is trained within the function described in Section 7 (see also Section 8).

In [ ]:
def build_transformer_baseline(seq_len, n_features, num_heads, n_blocks, d_model, ff_dim, dropout):
    inp = Input(shape=(seq_len, n_features), name="input_sequence")
    x = Dense(d_model, name="input_projection")(inp); x = PositionalEncoding(name="pos_enc")(x); x = Dropout(dropout)(x)
    for _ in range(n_blocks): x = transformer_block(x, num_heads, d_model, ff_dim, dropout)
    x = GlobalAveragePooling1D()(x); x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x); x = Dense(32, activation="relu")(x)
    return Model(inputs=inp, outputs=Dense(1, activation="linear")(x), name="Transformer_Baseline")

## 5.5 Regime-specific expert heads and Soft-Routing MoE Transformer

`make_expert_cfgs` defines the configuration of the expert heads (units, number of layers and dropout per regime). `build_sparse_moe_transformer` builds the Transformer backbone shared by all experts, one dense head per regime, and the fusion layer that weights the expert outputs with the routing probabilities (soft, dense routing).

In [ ]:
def make_expert_cfgs(n):
    cfgs = []
    for i in range(n):
        ratio = i / max(n-1, 1)
        units = max(16, int(64*(1-0.5*ratio)))
        n_layers = 2 if ratio < 0.5 else 1
        drop = round(0.1+0.2*ratio, 2)
        cfgs.append({"units": units, "n_layers": n_layers, "dropout": drop})
    return cfgs

In [ ]:
def build_sparse_moe_transformer(seq_len, n_features, n_experts, expert_cfgs, num_heads, n_blocks, d_model, ff_dim, dropout):
    inp_seq = Input(shape=(seq_len, n_features), name="input_sequence")
    inp_routing = Input(shape=(n_experts,), name="input_routing_probs")
    x = Dense(d_model, name="input_projection")(inp_seq); x = PositionalEncoding(name="pos_enc")(x); x = Dropout(dropout)(x)
    for _ in range(n_blocks): x = transformer_block(x, num_heads, d_model, ff_dim, dropout)
    x = GlobalAveragePooling1D()(x); x = Dense(64, activation="relu")(x); x = Dropout(dropout)(x)
    expert_outputs = []
    for i, cfg in enumerate(expert_cfgs):
        h = x
        for li in range(cfg["n_layers"]):
            h = Dense(cfg["units"], activation="relu", name=f"expert_{i}_h{li}")(h)
            if cfg.get("dropout",0) > 0: h = Dropout(cfg["dropout"], name=f"expert_{i}_drop{li}")(h)
        expert_outputs.append(Dense(1, activation="linear", name=f"expert_{i}_out")(h))
    weighted_output = MoEMergeLayer(n_experts)(expert_outputs + [inp_routing])
    return Model(inputs=[inp_seq, inp_routing], outputs=weighted_output, name="SparseMoE_Transformer")

# 6. Hard-Routing HMM-MoE

Regime-aware benchmark with heterogeneous experts. The block implements:

- partition of the training data by HMM state and construction of one data subset per regime;
- grid search, for every regime, over MLP, LSTM, BiLSTM and Transformer configurations, with selection of the best configuration by validation MSE (with an optional cache of the selected configurations);
- training of an LSTM gater with the HMM state labels;
- deterministic routing at inference: the expert associated with the highest-probability regime (argmax) produces the forecast;
- batch prediction, reconstruction of prices from predicted log-returns and computation of the evaluation metrics.

## 6.1 Hyperparameter grids of the experts

In [ ]:
PARAM_GRID_CLASSIC = {"model_type": ["LSTM","MLP","BiLSTM"], "layers": [1,2], "neurons": [32,64], "activation": ["tanh","relu"]}
PARAM_GRID_TRANSFORMER = {"model_type": ["Transformer"], "num_heads": [2,4], "n_blocks": [1,2,3], "d_model": [64], "ff_dim": [128], "dropout": [0.1]}
PARAM_GRID_CLASSIC_FAST = {"model_type": ["LSTM","MLP","BiLSTM"], "layers": [1], "neurons": [64], "activation": ["tanh"]}
PARAM_GRID_TRANSFORMER_FAST = {"model_type": ["Transformer"], "num_heads": [4], "n_blocks": [2], "d_model": [64], "ff_dim": [128], "dropout": [0.1]}

if GRID_FAST:
    GRIDS_HARD = list(ParameterGrid(PARAM_GRID_CLASSIC_FAST)) + list(ParameterGrid(PARAM_GRID_TRANSFORMER_FAST))
    print(f"GRID_FAST activado: {len(GRIDS_HARD)} configs")
else:
    GRIDS_HARD = list(ParameterGrid(PARAM_GRID_CLASSIC)) + list(ParameterGrid(PARAM_GRID_TRANSFORMER))
    print(f"Grid completo: {len(GRIDS_HARD)} configs")

GRID_CACHE = {}

## 6.2 Hard-routing benchmark: training and evaluation

In [ ]:
def entrenar_benchmark_hard(df, nombre, seq_length=SEQ_LENGTH, epochs=EPOCHS,
                             batch_size=BATCH_SIZE, min_obs_por_estado=10, entrenar_gater=True, seed=42):
    set_seed(seed)
    print(f"\n{'='*60}\nBenchmark hard-routing: {nombre} (seed={seed})\n{'='*60}")
    df = df.copy()
    split_rows = int((df["Split"]=="train").sum())
    scaler = StandardScaler()
    df["Return_scaled"] = np.nan
    df.loc[:split_rows-1, "Return_scaled"] = scaler.fit_transform(df.loc[:split_rows-1, ["Log_Return"]])
    df.loc[split_rows:, "Return_scaled"] = scaler.transform(df.loc[split_rows:, ["Log_Return"]])
    states = sorted(df["State"].unique()); n_experts = len(states)
    state_to_expert = {s: i for i, s in enumerate(states)}
    full = df["Return_scaled"].values
    X_all, y_all = create_sequences(full, seq_length)
    X_all = X_all.reshape(-1, seq_length, 1)
    split_seq = split_rows - seq_length
    X_train, X_test = X_all[:split_seq], X_all[split_seq:]
    y_train, y_test = y_all[:split_seq], y_all[split_seq:]
    df_train = df.iloc[:split_rows]
    expert_data = {}
    for state in states:
        vals = df_train.loc[df_train["State"]==state, "Return_scaled"].values
        if len(vals) > seq_length:
            Xs, ys = create_sequences(vals, seq_length)
            expert_data[state_to_expert[state]] = (Xs.reshape(-1, seq_length, 1), ys)
    experts, expert_cfgs = [None]*n_experts, [None]*n_experts
    for idx in range(n_experts):
        if idx not in expert_data: continue
        X_e, y_e = expert_data[idx]
        split_e = int(0.8*len(X_e))
        if split_e < 10: X_tr_e, y_tr_e, X_val_e, y_val_e = X_e, y_e, X_e, y_e
        else: X_tr_e, y_tr_e, X_val_e, y_val_e = X_e[:split_e], y_e[:split_e], X_e[split_e:], y_e[split_e:]
        best_mse, best_model, best_cfg = np.inf, None, None
        cache_key = (nombre, idx)
        if CACHE_GRID_SEARCH and cache_key in GRID_CACHE:
            best_cfg = GRID_CACHE[cache_key]
            model = build_model_from_cfg(best_cfg, seq_length)
            lr = 5e-4 if best_cfg["model_type"]=="Transformer" else 1e-3
            model.compile(optimizer=Adam(lr), loss="mse")
            model.fit(X_tr_e, y_tr_e, epochs=epochs, batch_size=batch_size, verbose=0,
                      callbacks=[EarlyStopping(patience=5, restore_best_weights=True, monitor="loss")])
            best_model = model
            best_mse = mean_squared_error(y_val_e, model.predict(X_val_e, verbose=0))
            print(f"Experto {idx}: cache {best_cfg['model_type']} -> MSE={best_mse:.6f}")
        else:
            print(f"\nGrid search experto {idx} -- {len(GRIDS_HARD)} configs")
            for cfg in GRIDS_HARD:
                model = build_model_from_cfg(cfg, seq_length)
                lr = 5e-4 if cfg["model_type"]=="Transformer" else 1e-3
                model.compile(optimizer=Adam(lr), loss="mse")
                model.fit(X_tr_e, y_tr_e, epochs=epochs, batch_size=batch_size, verbose=0,
                          callbacks=[EarlyStopping(patience=5, restore_best_weights=True, monitor="loss")])
                mse_e = mean_squared_error(y_val_e, model.predict(X_val_e, verbose=0))
                if mse_e < best_mse: best_mse, best_model, best_cfg = mse_e, model, cfg
            if CACHE_GRID_SEARCH: GRID_CACHE[cache_key] = best_cfg
            print(f"Experto {idx}: {best_cfg} -> MSE={best_mse:.6f}")
        experts[idx], expert_cfgs[idx] = best_model, best_cfg
    gater = None
    if entrenar_gater:
        gater = build_gater(seq_length, n_experts)
        gater.compile(optimizer=Adam(1e-3), loss="categorical_crossentropy")
        train_states = df["State"].values[seq_length:split_rows]
        y_gater = np.zeros((len(train_states), n_experts))
        for i, s in enumerate(train_states): y_gater[i, state_to_expert[s]] = 1
        gater.fit(X_train, y_gater, epochs=epochs, batch_size=batch_size, verbose=0,
                  callbacks=[EarlyStopping(patience=3, restore_best_weights=True, monitor="loss")])
    # Predicción en BATCH
    if entrenar_gater:
        all_probs = gater.predict(X_test, verbose=0)
        choices = np.argmax(all_probs, axis=1)
    else:
        choices = np.array([state_to_expert[int(df.iloc[split_rows+i]["State"])] for i in range(len(X_test))])
    fallback = next(j for j in range(n_experts) if experts[j] is not None)
    choices = np.where(np.array([experts[c] is not None for c in choices]), choices, fallback)
    preds = np.zeros(len(X_test), dtype=np.float32)
    used = choices.tolist()
    for exp_idx in range(n_experts):
        if experts[exp_idx] is None: continue
        mask = (choices == exp_idx)
        if mask.sum() == 0: continue
        preds[mask] = experts[exp_idx].predict(X_test[mask], verbose=0).flatten()
    true_ret = scaler.inverse_transform(y_test.reshape(-1,1)).flatten()
    pred_ret = scaler.inverse_transform(preds.reshape(-1,1)).flatten()
    base_prices = df["Price"].values[split_rows-1:split_rows-1+len(pred_ret)]
    prices_real = base_prices * np.exp(true_ret)
    prices_pred = base_prices * np.exp(pred_ret)
    dates_test = df["Date"].values[split_rows:split_rows+len(pred_ret)]
    metrics = {"dataset": nombre, "modelo": "HMM-MoE (hard routing)",
        "MSE_price": mean_squared_error(prices_real, prices_pred),
        "MAE_price": mean_absolute_error(prices_real, prices_pred),
        "RMSE_price": sqrt(mean_squared_error(prices_real, prices_pred)),
        "R2_price": r2_score(prices_real, prices_pred),
        "RMSE_return": sqrt(mean_squared_error(true_ret, pred_ret)),
        "R2_return": r2_score(true_ret, pred_ret)}
    print(pd.Series(metrics))
    return metrics, expert_cfgs, used, dates_test, prices_real, prices_pred, states

# 7. Soft-Routing Mixture of Experts

Proposed architecture: a Transformer backbone shared by all experts with regime-specific expert heads, whose outputs are fused through a probability-weighted combination using the routing probabilities estimated by the gater (every expert contributes to every forecast). The training function:

1. builds the multivariate features (normalised returns, rolling volatility, RSI and one-hot HMM states);
2. reserves the final 15% of the training sequences as a chronological validation subset;
3. pre-trains the LSTM gater with the HMM labels and obtains the routing probabilities;
4. trains the global Transformer baseline (Section 8);
5. trains the Soft-Routing MoE Transformer end to end;
6. reconstructs the prices and computes the metrics of both models, and returns the dominant expert of each test observation.

In the code this model is identified as "Sparse/Soft MoE" in several identifiers (see Appendix).

In [ ]:
TRANS_CFG = dict(num_heads=4, n_blocks=3, d_model=64, ff_dim=256, dropout=0.1)

def entrenar_sparse_moe(df, nombre, seq_length=SEQ_LENGTH, pred_len=1,
                         epochs=100, batch_size=BATCH_SIZE, trans_cfg=TRANS_CFG, seed=42):
    set_seed(seed)
    print(f"\n{'='*60}\nSparse/Soft MoE Transformer: {nombre} (seed={seed})\n{'='*60}")
    df = df.copy()
    states = sorted(df["State"].unique()); n_experts = len(states)
    state_to_expert = {s: i for i, s in enumerate(states)}
    expert_cfgs = make_expert_cfgs(n_experts)
    split_rows = int((df["Split"]=="train").sum())
    split_seq = split_rows - seq_length
    ret_train_raw = df["Log_Return"].values[:split_rows]
    ret_min, ret_max = ret_train_raw.min(), ret_train_raw.max()
    def norm(x): return (x - ret_min) / (ret_max - ret_min)
    def denorm(x): return x * (ret_max - ret_min) + ret_min
    df["Return_scaled"] = norm(df["Log_Return"].values)
    df["Volatility"] = df["Return_scaled"].rolling(window=5).std().fillna(0)
    vol_max = df["Volatility"].iloc[:split_rows].max()
    df["Volatility"] = df["Volatility"] / (vol_max if vol_max > 0 else 1)
    df["RSI"] = calculate_rsi(df["Price"], window=14).fillna(50) / 100.0
    feature_cols = ["Return_scaled", "Volatility", "RSI"]
    for state in states:
        col = f"HMM_{state}"; df[col] = (df["State"]==state).astype(float); feature_cols.append(col)
    n_features = len(feature_cols)
    X_all, y_all = create_sequences_multivariate(df[feature_cols], df["Return_scaled"], seq_length, pred_len)
    X_train, X_test = X_all[:split_seq], X_all[split_seq:]
    y_train, y_test = y_all[:split_seq], y_all[split_seq:]
    n_val = int(len(X_train)*0.15)
    X_tr, y_tr = X_train[:-n_val], y_train[:-n_val]
    X_val, y_val = X_train[-n_val:], y_train[-n_val:]
    # Gater
    gater = build_gater(seq_length, n_experts, n_features)
    gater.compile(optimizer=Adam(1e-3), loss="categorical_crossentropy")
    train_states = df["State"].values[seq_length:split_rows]
    y_gater = np.zeros((len(train_states), n_experts))
    for i, s in enumerate(train_states): y_gater[i, state_to_expert[s]] = 1
    y_gater_tr, y_gater_val = y_gater[:-n_val], y_gater[-n_val:]
    gater.fit(X_tr, y_gater_tr, validation_data=(X_val, y_gater_val),
              epochs=50, batch_size=batch_size, verbose=0,
              callbacks=[EarlyStopping(patience=5, restore_best_weights=True, monitor="val_loss")])
    probs_tr = gater.predict(X_tr, verbose=0)
    probs_val = gater.predict(X_val, verbose=0)
    probs_test = gater.predict(X_test, verbose=0)
    # Transformer baseline
    model_base = build_transformer_baseline(seq_length, n_features, **trans_cfg)
    model_base.compile(optimizer=Adam(5e-4), loss="mse")
    model_base.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=epochs, batch_size=batch_size, verbose=0,
                   callbacks=[EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
                              ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)])
    # Sparse/Soft MoE
    model_moe = build_sparse_moe_transformer(seq_length, n_features, n_experts, expert_cfgs, **trans_cfg)
    model_moe.compile(optimizer=Adam(5e-4), loss="mse")
    model_moe.fit([X_tr, probs_tr], y_tr, validation_data=([X_val, probs_val], y_val),
                  epochs=epochs, batch_size=batch_size, verbose=0,
                  callbacks=[EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
                             ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)])
    preds_base = model_base.predict(X_test, verbose=0).flatten()
    preds_moe = model_moe.predict([X_test, probs_test], verbose=0).flatten()
    true_ret = denorm(y_test); pred_ret_base = denorm(preds_base); pred_ret_moe = denorm(preds_moe)
    base_prices = df["Price"].values[split_rows-1:split_rows-1+len(pred_ret_base)]
    prices_real = base_prices * np.exp(true_ret)
    prices_base = base_prices * np.exp(pred_ret_base)
    prices_moe = base_prices * np.exp(pred_ret_moe)
    dates_test = df["Date"].values[split_rows:split_rows+len(pred_ret_base)]
    metrics_base = {"dataset": nombre, "modelo": "Transformer (global)",
        "MSE_price": mean_squared_error(prices_real, prices_base), "MAE_price": mean_absolute_error(prices_real, prices_base),
        "RMSE_price": sqrt(mean_squared_error(prices_real, prices_base)), "R2_price": r2_score(prices_real, prices_base),
        "RMSE_return": sqrt(mean_squared_error(true_ret, pred_ret_base)), "R2_return": r2_score(true_ret, pred_ret_base)}
    metrics_moe = {"dataset": nombre, "modelo": "Soft-Routing MoE Transformer",
        "MSE_price": mean_squared_error(prices_real, prices_moe), "MAE_price": mean_absolute_error(prices_real, prices_moe),
        "RMSE_price": sqrt(mean_squared_error(prices_real, prices_moe)), "R2_price": r2_score(prices_real, prices_moe),
        "RMSE_return": sqrt(mean_squared_error(true_ret, pred_ret_moe)), "R2_return": r2_score(true_ret, pred_ret_moe)}
    print(pd.DataFrame([metrics_base, metrics_moe]).to_string(index=False))
    used_moe = np.argmax(probs_test, axis=1)
    return metrics_base, metrics_moe, dates_test, prices_real, prices_base, prices_moe, used_moe, states

# 8. Global Transformer Benchmark

The Global Transformer benchmark is integrated directly into the Soft-Routing training pipeline:

- **Architecture:** `build_transformer_baseline` (Section 5.4), a Transformer encoder with the same backbone configuration (`TRANS_CFG`) as the Soft-Routing MoE but without regime-specific heads.
- **Training and evaluation:** inside `entrenar_sparse_moe` (Section 7), on the same features, training/validation split and optimisation settings as the Soft-Routing MoE. Its metrics are returned under the label `"Transformer (global)"`.
- **Comparison:** the Diebold–Mariano test of Section 9 compares the predictions of this model with those of the Soft-Routing MoE on the price scale.

# 9. Evaluation Metrics and Statistical Functions

Evaluation, statistical-comparison and aggregation functions, followed by the script that drives the main experiments.

**Evaluation metrics.** The metrics are computed inside the training functions of Sections 6 and 7 using scikit-learn: MSE, MAE, RMSE and R² on the reconstructed price scale, and RMSE and R² on the return scale. Predicted prices are reconstructed as P̂ₜ = Pₜ₋₁ · exp(r̂ₜ), anchored on the last observed price (non-recursive).

## 9.1 Diebold–Mariano test

Diebold–Mariano test of equal predictive accuracy between two forecasts (`pred_a`, `pred_b`) with loss defined by the power of the absolute error (squared error by default). A positive statistic indicates that `pred_a` has the larger error.

In [ ]:
def diebold_mariano(y_true, pred_a, pred_b, h=1, power=2):
    from scipy.stats import t as t_dist
    e_a = np.asarray(y_true) - np.asarray(pred_a)
    e_b = np.asarray(y_true) - np.asarray(pred_b)
    d = np.abs(e_a)**power - np.abs(e_b)**power
    n = len(d); d_mean = d.mean()
    gamma0 = np.mean((d - d_mean)**2)
    gammas = [np.mean((d[k:]-d_mean)*(d[:-k]-d_mean)) for k in range(1, h)]
    var_d = (gamma0 + 2*np.sum(gammas)) / n
    if var_d <= 0: return np.nan, np.nan
    dm_stat = d_mean / np.sqrt(var_d)
    adj = np.sqrt((n+1-2*h+h*(h-1)/n) / n)
    dm_stat *= adj
    p_value = 2 * t_dist.cdf(-abs(dm_stat), df=n-1)
    return dm_stat, p_value

## 9.2 Aggregation of metrics across seeds

In [ ]:
def agregar_metricas(registros):
    df = pd.DataFrame(registros)
    metric_cols = [c for c in df.columns if c not in ("dataset","modelo","seed")]
    agg = df.groupby(["dataset","modelo"])[metric_cols].agg(["mean","std"])
    return df, agg

## 9.3 Plotting functions for the results

Comparison plot between the real price, the global Transformer and the Soft-Routing MoE, and plot of expert utilisation in the test period.

In [ ]:
def graficar_comparacion(dates_test, prices_real, prices_base, prices_moe, nombre):
    """Gráfico final: Precio Real vs Transformer Baseline vs Sparse/Soft MoE (precios)."""
    plt.figure(figsize=(14, 5))
    plt.plot(dates_test, prices_real, label="Real", color="black", linewidth=1.6)
    plt.plot(dates_test, prices_base, label="Transformer (global baseline)", color="#d62728", alpha=0.85, linestyle="--")
    plt.plot(dates_test, prices_moe, label="Soft-Routing MoE Transformer", color="#1f77b4", alpha=0.85)       #He cambiado label="Sparse/Soft MoE Transformer" x "Soft-Routing MoE Transformer"
    plt.title(f"Predicted vs. Actual Price (Test Period) -- {nombre}")
    plt.xlabel("Date"); plt.ylabel("Price ($)")
    plt.legend(fontsize=11); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
def graficar_estados_usados(used_hard, used_soft, states, nombre):
    n_experts = len(states); idx = np.arange(n_experts)
    cont_hard = np.array([(np.asarray(used_hard)==i).sum() for i in idx])
    cont_soft = np.array([(np.asarray(used_soft)==i).sum() for i in idx])
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
    etiquetas = [f"Exp {i}\n(estado {states[i]})" for i in idx]
    for ax, cont, titulo, color in [(axes[0],cont_hard,"Hard routing","#d62728"),(axes[1],cont_soft,"Soft-Routing MoE (dominant)","#1f77b4")]:
        barras = ax.bar(idx, cont, color=color, alpha=0.85)
        ax.set_xticks(idx); ax.set_xticklabels(etiquetas); ax.set_title(titulo); ax.set_xlabel("Experto")
        total = cont.sum()
        for b, c in zip(barras, cont):
            pct = 100*c/total if total>0 else 0
            ax.text(b.get_x()+b.get_width()/2, b.get_height(), f"{c}\n({pct:.0f}%)", ha="center", va="bottom", fontsize=9)
    axes[0].set_ylabel("Veces como experto dominante en test")
    fig.suptitle(f"Uso de expertos en test -- {nombre}"); plt.tight_layout(); plt.show()
    return pd.DataFrame({"estado": [states[i] for i in idx], "hard_dominante": cont_hard, "soft_dominante": cont_soft})

## 9.4 Main execution script

> **Execution configuration.**
> The following block preserves the configuration used for this experimental execution.

This script runs the full pipeline (HMM, Hard-Routing HMM-MoE, global Transformer and Soft-Routing MoE, with the Diebold–Mariano comparison) over the datasets and seeds defined in `DATASETS` and `SEEDS` (Section 1.3).

In [ ]:
resultados_hmm = {}
registros_por_semilla = []
dm_resultados = []

for csv_file, nombre in DATASETS:
    print(f"\n\n{'#'*70}\n# {nombre}\n{'#'*70}")
    df_precio = cargar_precio(csv_file)
    df_feat = construir_features_hmm(df_precio)
    df_states, hmm_model, hmm_scaler, split_rows, bic_df = ajustar_hmm_sin_fuga(df_feat)
    df_states.to_csv(f"{nombre}_HMM_States_corregido.csv", index=False)
    resultados_hmm[nombre] = (df_states, hmm_model, split_rows)
    graficar_estados(df_states, nombre)
    analizar_frecuencia_duracion(df_states, titulo=nombre)
    analizar_frecuencia_duracion(df_states[df_states["Split"]=="train"], titulo=f"{nombre} (solo train)")

    seeds_ordenadas = ([SEED_GRID]+[s for s in SEEDS if s!=SEED_GRID] if SEED_GRID in SEEDS else SEEDS)
    graf_guardado = None
    uso_guardado = None
    for seed in seeds_ordenadas:
        (metrics_hard, cfgs_hard, used_hard,
         dates_test_h, prices_real_h, prices_pred_h, states_h) = entrenar_benchmark_hard(df_states, nombre, seed=seed)
        (metrics_base, metrics_moe,
         dates_test_m, prices_real_m, prices_base_m, prices_pred_m,
         used_moe, states_m) = entrenar_sparse_moe(df_states, nombre, seed=seed)

        for m in (metrics_hard, metrics_base, metrics_moe):
            m = dict(m); m["seed"] = seed
            registros_por_semilla.append(m)

        # Diebold-Mariano: Transformer baseline vs Sparse/Soft MoE (sobre PRECIOS)
        dm_stat, dm_p = diebold_mariano(prices_real_m, prices_base_m, prices_pred_m)
        dm_resultados.append({"dataset": nombre, "seed": seed, "DM_stat": dm_stat, "p_value": dm_p})

        if graf_guardado is None:
            # Guardamos Transformer baseline vs MoE (sobre PRECIOS) para el grafico final
            graf_guardado = (dates_test_m, prices_real_m, prices_base_m, prices_pred_m)
            uso_guardado = (used_hard, used_moe, states_m)

    # Grafico final: Real vs Transformer Baseline vs Sparse/Soft MoE (PRECIOS)
    graficar_comparacion(*graf_guardado, nombre)

    tabla_uso = graficar_estados_usados(*uso_guardado, nombre)
    print(f"\nUso de expertos en test -- {nombre}:")
    print(tabla_uso.to_string(index=False))

# Agregacion
df_crudo, tabla_agg = agregar_metricas(registros_por_semilla)
print("\n\n" + "="*70)
print(f"RESULTADOS AGREGADOS SOBRE {len(SEEDS)} SEMILLAS (media +/- std)")
print("="*70)
print(tabla_agg.round(4).to_string())
df_crudo.to_csv("resultados_por_semilla.csv", index=False)
tabla_agg.round(6).to_csv("resultados_agregados.csv")

dm_df = pd.DataFrame(dm_resultados)
print("\n" + "="*70)
print("TEST DIEBOLD-MARIANO: Transformer baseline vs. Sparse/Soft MoE (por semilla)")
print("DM>0 => baseline tiene mas error (gana MoE); p<0.05 => diferencia significativa")
print("="*70)
print(dm_df.round(4).to_string(index=False))
dm_resumen = dm_df.groupby("dataset").agg(
    DM_stat_medio=("DM_stat", "mean"),
    p_value_medio=("p_value", "mean"),
    n_signif=("p_value", lambda s: int((s < 0.05).sum())),
)
print("\nResumen por activo:")
print(dm_resumen.round(4).to_string())
dm_df.to_csv("diebold_mariano_por_semilla.csv", index=False)

# 10. Random Walk Benchmark

## Deterministic one-step-ahead persistence benchmark

The Random Walk benchmark predicts the next price as the last observed price,

$$\hat{P}_t = P_{t-1},$$

so that the equivalent predicted log-return is zero, $\hat{r}_t = 0$. The block **applies** this benchmark as a predictive reference over existing series: it does not generate new Random Walk series. For each uploaded series it computes the log-returns, takes the test partition (using the `Split` column if the file contains it, and the train ratio otherwise), verifies that the reconstruction `base · exp(ret)` coincides with the observed prices, and reports MAE, RMSE, MSE and R² on the price scale and on the return scale.

This block is self-contained: it declares its own imports and helper functions (`cargar`, `metricas`, `random_walk`) and does not depend on the variables of the main pipeline.

> **Execution-specific configuration.**
> The series files are uploaded interactively (`files.upload()`, Google Colab), so the series evaluated are those selected at execution time. The window and split parameters defined below are preserved without modification for traceability.

In [ ]:
import numpy as np, pandas as pd
from google.colab import files
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TRAIN_RATIO, VOL_WINDOW = 0.80, 5

def cargar(path):
    df = pd.read_excel(path) if path.lower().endswith(("xlsx", "xls")) else pd.read_csv(path)
    f = next(c for c in df.columns if c.strip().lower() in ("date", "fecha"))
    p = next(c for c in df.columns if c.strip() in ("Price", "Close", "Adj Close", "close", "price"))
    s = next((c for c in df.columns if c.strip().lower() == "split"), None)
    out = pd.DataFrame({"Date": pd.to_datetime(df[f]), "Price": pd.to_numeric(df[p], errors="coerce")})
    if s: out["Split"] = df[s].astype(str).str.strip().str.lower()
    out = out.dropna().sort_values("Date").reset_index(drop=True)
    assert (out["Price"] > 0).all() and out["Date"].is_unique, f"{path}: precios <= 0 o fechas duplicadas"
    return out

def metricas(y, yhat):
    mse = mean_squared_error(y, yhat)
    return dict(MAE=mean_absolute_error(y, yhat), RMSE=np.sqrt(mse), MSE=mse, R2=r2_score(y, yhat))

def random_walk(raw):
    df = raw.copy()
    df["Log_Return"] = np.log(df["Price"] / df["Price"].shift(1))
    df["Vol"] = df["Log_Return"].rolling(VOL_WINDOW).std()
    df = df.dropna().reset_index(drop=True)
    n = len(df)
    s = int((df["Split"] == "train").sum()) if "Split" in df else int(TRAIN_RATIO * n)
    test = df.iloc[s:]
    ret, real = test["Log_Return"].values, test["Price"].values
    base = df["Price"].values[s-1:s-1+len(ret)]          # precio real del instante anterior
    assert len(base) == len(ret) == len(real)
    assert np.allclose(base * np.exp(ret), real, rtol=1e-9)
    return metricas(real, base), metricas(ret, np.zeros_like(ret)), len(test)

res_p, res_r = [], []
for f in files.upload():
    nombre = f.rsplit(".", 1)[0]
    mp, mr, nt = random_walk(cargar(f))
    res_p.append({"Serie": nombre, "N_test": nt, **mp})
    res_r.append({"Serie": nombre, "N_test": nt, **mr})

tabla_precios = pd.DataFrame(res_p).round(4)
tabla_retornos = pd.DataFrame(res_r).round(6)
print("RANDOM WALK - PRECIOS");   print(tabla_precios.to_string(index=False))
print("\nRANDOM WALK - RETORNOS"); print(tabla_retornos.to_string(index=False))

# 11. Additional Multi-Seed and Statistical Analysis

## Supplementary / additional analysis


> **Development / supplementary implementation.**
> This block corresponds to an additional experimental implementation retained for methodological traceability.

This section collects a supplementary analysis that is **not a required part of the main pipeline**. It re-trains only the Soft-Routing MoE Transformer (labelled *Sparse/Soft MoE Transformer* in the code and in the output files) with **seven additional random seeds**, complementing the three seeds of the main pipeline for a total of ten runs per series. Its objectives are to characterise the run-to-run variance of the model, to obtain robust means over n = 10 runs, and to apply the Mann–Whitney U test (n = 10 versus n = 3) against reference baselines, whose per-seed results are loaded from file.

The block is self-contained: it declares its own dependencies, imports, configuration and function definitions (some of them analogous to those of Sections 3–5 and 7), and it is designed to be executed once per series, accumulating the results in one sub-folder per dataset; a final cell consolidates all the datasets.

The technical settings follow those of the main pipeline: leakage-free HMM (estimated once on the training partition), features Return_scaled + Volatility + RSI + one-hot HMM states, Adam(5e-4), MSE loss, batch size 32, EarlyStopping with patience 15, and the same Transformer configuration.

## 11.1 Installation, imports and global configuration

The HMM library is installed (same dependency as the main pipeline) and imports and global configuration are loaded.

**Seeds**
- Seeds of the main pipeline: `[42, 123, 2024]`
- Additional seeds trained in this block: `[7, 13, 100, 500, 999, 1234, 5000]`
- Total: **10 runs of the Soft-Routing MoE per dataset**

> **Execution configuration.**
> The following block preserves the configuration used for this experimental execution.

In [ ]:
!pip install hmmlearn --quiet

In [ ]:
import os
import warnings
import json
from pathlib import Path
from collections import defaultdict
from math import sqrt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Flatten,
    MultiHeadAttention, Dropout, LayerNormalization, GlobalAveragePooling1D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from hmmlearn.hmm import GaussianHMM
from scipy.stats import multivariate_normal, mannwhitneyu, wilcoxon, ttest_rel
from scipy.special import logsumexp

warnings.filterwarnings("ignore")


# ---- MISMA CONFIGURACION QUE EL PIPELINE ORIGINAL ----
SEQ_LENGTH  = 30
TRAIN_RATIO = 0.80
EPOCHS      = 100
BATCH_SIZE  = 32

HMM_MIN_STATES = 2
HMM_MAX_STATES = 7
HMM_VOL_WINDOW = 5
HMM_COV_TYPE   = "full"

TRANS_CFG = dict(num_heads=4, n_blocks=3, d_model=64, ff_dim=256, dropout=0.1)

# ---- SEMILLAS ----
SEEDS_ANTIGUAS = [42, 123, 2024]
SEEDS_NUEVAS   = [7, 13, 100, 500, 999, 1234, 5000]
SEEDS_TODAS    = SEEDS_ANTIGUAS + SEEDS_NUEVAS

# ---- DIRECTORIO RAIZ DE SALIDA (los resultados se guardan en subcarpetas por dataset) ----
OUT_DIR = "resultados_variance_sparse"
os.makedirs(OUT_DIR, exist_ok=True)


def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    np.random.seed(seed)
    import random
    random.seed(seed)
    tf.random.set_seed(seed)


print(f"Semillas antiguas: {SEEDS_ANTIGUAS}")
print(f"Semillas nuevas:   {SEEDS_NUEVAS}")
print(f"Total corridas del Sparse MoE por dataset: {len(SEEDS_TODAS)}")

## 11.2 Interactive file upload

The upload cell requests four files, one at a time, so that they can reside in different folders:

- **Step 1/4:** price CSV of the dataset to be processed (e.g. `NVIDIA.csv`, `APPLE.csv`, `SP500.csv`). The dataset name is derived from the file name, in upper case.
- **Step 2/4:** `resultados_por_semilla.csv`, produced by the main pipeline.
- **Step 3/4:** `metricas_por_semilla.csv`, with the per-seed metrics of the reference baselines.
- **Step 4/4:** CSV with the HMM states already computed for the dataset (e.g. `estados_hmm_NVIDIA.csv`). It must contain at least the columns `Date` and `State`, and preferably `Split` (train/test) in order to respect exactly the partition of the main pipeline.

Loading the HMM states, instead of re-estimating the HMM, ensures that all runs share exactly the same regime segmentation: small differences in library versions could otherwise produce slightly different states and affect the comparison between the three original seeds and the seven additional ones. In this way the only source of variability is the initialisation of the Soft-Routing MoE.

In each step, if the file is already available on disk (from a previous execution), the cell offers to reuse it; therefore, for subsequent series only the dataset CSV and its HMM states need to be uploaded. As the results files contain several series, the dataset name derived from the CSV file is validated against the names present in them; if no exact match is found, the available names are displayed and one of them can be selected.

> **Development / supplementary implementation.**
> This block corresponds to an additional experimental implementation retained for methodological traceability.

In [ ]:
def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _colab_upload_one(hint):
    """Un solo archivo via Colab. Devuelve el nombre subido o None si el usuario cancela."""
    from google.colab import files
    print(f"[Colab] {hint}")
    uploaded = files.upload()
    if not uploaded:
        print("  (sin subida)")
        return None
    if len(uploaded) > 1:
        print(f"  Aviso: subiste {len(uploaded)} archivos, uso el primero: {list(uploaded.keys())[0]}")
    return list(uploaded.keys())[0]


def _local_ask_path(hint):
    """Ruta local via input(). Devuelve la ruta o None si el usuario deja vacío."""
    ruta = input(f"[Local] Ruta a {hint} (Enter para omitir): ").strip()
    if not ruta:
        return None
    if not Path(ruta).exists():
        raise RuntimeError(f"No existe: {ruta}")
    return ruta


def obtener_archivo(paso_num, total_pasos, nombre_amigable, ejemplo, candidatos_disco=None):
    """Pide UN archivo. Si ya existe en disco (candidatos_disco), pregunta si reutilizarlo.
    Si el usuario no sube nada y hay candidato en disco, usa el candidato.
    """
    candidatos_disco = candidatos_disco or []
    print("\n" + "=" * 70)
    print(f"  PASO {paso_num}/{total_pasos}: {nombre_amigable}")
    print(f"  Ejemplo:                {ejemplo}")
    print("=" * 70)

    # Ver si ya hay algo en disco
    existente = next((c for c in candidatos_disco if Path(c).exists()), None)
    if existente:
        resp = input(f"  Ya existe '{existente}' en disco. Reutilizar? [S/n]: ").strip().lower()
        if resp in ("", "s", "si", "sí", "y", "yes"):
            print(f"  -> Reutilizado: {existente}")
            return existente

    # Pedir subida (Colab) o ruta (local)
    hint = f"Sube ahora: {nombre_amigable}"
    if _in_colab():
        subido = _colab_upload_one(hint)
    else:
        subido = _local_ask_path(nombre_amigable)

    if subido is None:
        if existente:
            print(f"  -> Sin subida, usando el existente: {existente}")
            return existente
        raise RuntimeError(f"Falta '{nombre_amigable}' (ni subido ni en disco)")

    return subido


def obtener_archivos_paso_a_paso():
    """Pide los cuatro archivos, uno por paso."""
    # 1) CSV del dataset a procesar
    dataset_csv = obtener_archivo(
        1, 4,
        "CSV del dataset a procesar (NVIDIA.csv, APPLE.csv, SP500.csv, ...)",
        "NVIDIA.csv",
        candidatos_disco=[],
    )
    dataset_nombre = Path(dataset_csv).stem.upper()

    # 2) resultados_por_semilla.csv del pipeline paper
    res_antiguas = obtener_archivo(
        2, 4,
        "resultados_por_semilla.csv (Sparse MoE del pipeline paper original)",
        "resultados_por_semilla.csv",
        candidatos_disco=["resultados_por_semilla.csv",
                           "./resultados_por_semilla.csv"],
    )

    # 3) metricas_por_semilla.csv de baselines modernos
    res_baseline = obtener_archivo(
        3, 4,
        "metricas_por_semilla.csv (baselines modernos DLinear/N-BEATS/PatchTST)",
        "metricas_por_semilla.csv",
        candidatos_disco=["metricas_por_semilla.csv",
                           "resultados_baselines_modernos/metricas_por_semilla.csv"],
    )

    # 4) CSV con estados HMM ya calculados (CRITICO para comparabilidad con las seeds antiguas)
    estados_hmm = obtener_archivo(
        4, 4,
        f"CSV con estados HMM ya calculados para {dataset_nombre} (mismos que usaron los otros modelos del paper)",
        f"estados_hmm_{dataset_nombre}.csv (debe tener columnas 'Date' y 'State')",
        candidatos_disco=[
            f"estados_hmm_{dataset_nombre}.csv",
            f"{dataset_nombre}_hmm_states.csv",
            f"hmm_states_{dataset_nombre}.csv",
            f"estados_hmm.csv",
        ],
    )

    return dataset_csv, dataset_nombre, res_antiguas, res_baseline, estados_hmm


def resolver_nombre_en_csv(csv_path, dataset_nombre_actual, columna="dataset"):
    """Comprueba si `dataset_nombre_actual` existe en la columna del CSV.
    Si no aparece exacto, muestra los nombres disponibles y pide al usuario
    que elija cual usar (heuristica: sugiere el que contenga el nombre actual
    como substring, ignorando mayusculas/espacios).
    """
    df = pd.read_csv(csv_path)
    if columna not in df.columns:
        raise KeyError(f"'{csv_path}' no tiene columna '{columna}'. Cols: {list(df.columns)}")

    nombres = sorted(df[columna].astype(str).unique())
    if dataset_nombre_actual in nombres:
        return dataset_nombre_actual

    # Sugerencia por substring case-insensitive
    key = dataset_nombre_actual.lower().replace(" ", "").replace("_", "")
    similares = [n for n in nombres
                  if key in n.lower().replace(" ", "").replace("_", "")
                  or n.lower().replace(" ", "").replace("_", "") in key]

    print(f"\n  Aviso: '{dataset_nombre_actual}' no aparece en '{csv_path}'.")
    print(f"  Nombres disponibles: {nombres}")
    if similares:
        print(f"  Coincidencias parciales: {similares}")
        resp = input(f"  Usar '{similares[0]}'? [S/n, o escribe otro nombre]: ").strip()
        if resp == "" or resp.lower() in ("s", "si", "sí", "y", "yes"):
            return similares[0]
        if resp in nombres:
            return resp
    else:
        resp = input(f"  Escribe el nombre exacto a usar (o Enter para abortar): ").strip()
        if resp in nombres:
            return resp

    raise RuntimeError(f"No se pudo resolver '{dataset_nombre_actual}' en '{csv_path}'.")

> **Execution configuration.**
> The following block preserves the configuration used for this experimental execution.

The cell below runs the interactive upload for the series being processed in each execution.

In [ ]:
# ============================================================
# Ejecutar el uploader
# ============================================================
DATASET_CSV, DATASET_NOMBRE, PATH_RES_ANTIGUAS, PATH_RES_BASELINE, PATH_ESTADOS_HMM = obtener_archivos_paso_a_paso()

# Validar / resolver nombre en los CSVs multi-dataset
print("\n" + "=" * 70)
print("  Validando nombre del dataset en los CSVs de resultados")
print("=" * 70)

NOMBRE_EN_ANTIGUAS = resolver_nombre_en_csv(PATH_RES_ANTIGUAS, DATASET_NOMBRE)
NOMBRE_EN_BASELINE = resolver_nombre_en_csv(PATH_RES_BASELINE, DATASET_NOMBRE)

print(f"\n  En '{PATH_RES_ANTIGUAS}' se usara: '{NOMBRE_EN_ANTIGUAS}'")
print(f"  En '{PATH_RES_BASELINE}' se usara: '{NOMBRE_EN_BASELINE}'")

# Subcarpeta de salida (por dataset)
DATASET_OUT_DIR = os.path.join(OUT_DIR, DATASET_NOMBRE)
os.makedirs(DATASET_OUT_DIR, exist_ok=True)

print(f"\n  Dataset a entrenar:   {DATASET_CSV}  ->  '{DATASET_NOMBRE}'")
print(f"  Estados HMM oficiales:{PATH_ESTADOS_HMM}")
print(f"  Resultados iran a:    {DATASET_OUT_DIR}/")

## 11.3 Data loading and base features

> **Development / supplementary implementation.**
> This block corresponds to an additional experimental implementation retained for methodological traceability.

In [ ]:
def cargar_precio(path_csv, col_fecha="Date", col_precio=None):
    df = pd.read_csv(path_csv, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]
    if col_precio is None:
        candidatos = ["Price", "Close", "Adj Close", "close", "price", "Adj_Close", "adj_close"]
        col_precio = next((c for c in candidatos if c in df.columns), None)
        if col_precio is None:
            raise KeyError(f"No encuentro columna de precio en {path_csv}. Columnas: {list(df.columns)}.")
        print(f"Columna de precio detectada en {path_csv}: '{col_precio}'")
    if col_fecha not in df.columns:
        candidatos_fecha = ["Date", "date", "Fecha", "DATE"]
        col_fecha_det = next((c for c in candidatos_fecha if c in df.columns), None)
        if col_fecha_det is None:
            raise KeyError(f"No encuentro columna de fecha en {path_csv}.")
        print(f"Columna de fecha detectada en {path_csv}: '{col_fecha_det}'")
        col_fecha = col_fecha_det
    df[col_fecha] = pd.to_datetime(df[col_fecha])
    df = df.sort_values(col_fecha).reset_index(drop=True)
    df = df[[col_fecha, col_precio]].dropna()
    df = df.rename(columns={col_fecha: "Date", col_precio: "Price"})
    return df


def construir_features_hmm(df, vol_window=HMM_VOL_WINDOW):
    df = df.copy()
    df["Log_Return"] = np.log(df["Price"] / df["Price"].shift(1))
    df["Volatility"] = df["Log_Return"].rolling(vol_window).std()
    df = df.dropna().reset_index(drop=True)
    return df


def create_sequences_multivariate(df_features, target_series, seq_len, pred_len=1):
    data = np.asarray(df_features.values, dtype=np.float32)
    targets = np.asarray(target_series.values, dtype=np.float32)
    n = len(data) - seq_len - pred_len + 1
    if n <= 0:
        return (np.empty((0, seq_len, data.shape[1]), dtype=np.float32),
                np.empty(0, dtype=np.float32))
    n_feat = data.shape[1]
    shape = (n, seq_len, n_feat)
    strides = (data.strides[0], data.strides[0], data.strides[1])
    X = np.lib.stride_tricks.as_strided(data, shape=shape, strides=strides).copy()
    y = targets[seq_len + pred_len - 1: seq_len + pred_len - 1 + n].copy()
    return X, y


def calculate_rsi(series, window=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

## 11.4 HMM without data leakage (auxiliary functions)

These functions replicate the HMM estimation of the main pipeline (Section 4) and are included for traceability; they are **not executed in this block**. The execution cell (Section 11.7) loads the HMM states directly from the CSV file uploaded in Step 4, which ensures that the same segmentation as in the main pipeline is used.

> **Development / supplementary implementation.**
> This block corresponds to an additional experimental implementation retained for methodological traceability.

In [ ]:
def compute_log_emissions(model, X):
    n_states = model.n_components
    return np.column_stack([
        multivariate_normal.logpdf(X, mean=model.means_[k], cov=model.covars_[k])
        for k in range(n_states)
    ])


def forward_filter(model, X):
    log_B = compute_log_emissions(model, X)
    n_obs, n_states = log_B.shape
    log_A = np.log(model.transmat_ + 1e-300)
    log_pi = np.log(model.startprob_ + 1e-300)
    log_alpha = np.zeros((n_obs, n_states))
    log_alpha[0] = log_pi + log_B[0]
    log_alpha[0] -= logsumexp(log_alpha[0])
    for t in range(1, n_obs):
        log_alpha[t] = logsumexp(log_alpha[t - 1][:, None] + log_A, axis=0) + log_B[t]
        log_alpha[t] -= logsumexp(log_alpha[t])
    return np.argmax(log_alpha, axis=1), np.exp(log_alpha)


def seleccionar_n_estados_train(X_train_scaled, min_states=HMM_MIN_STATES,
                                 max_states=HMM_MAX_STATES, cov_type=HMM_COV_TYPE,
                                 random_state=42):
    best_model, best_bic, best_n = None, np.inf, None
    resultados = []
    for n_states in range(min_states, max_states + 1):
        model = GaussianHMM(n_components=n_states, covariance_type=cov_type,
                             n_iter=1000, random_state=random_state)
        model.fit(X_train_scaled)
        ll = model.score(X_train_scaled)
        nf = X_train_scaled.shape[1]
        n_params = n_states ** 2 + 2 * n_states * nf - 1
        bic = -2 * ll + n_params * np.log(len(X_train_scaled))
        aic = -2 * ll + 2 * n_params
        resultados.append((n_states, bic, aic))
        if bic < best_bic:
            best_bic, best_model, best_n = bic, model, n_states
    print(f"Numero de estados seleccionado (BIC, solo train): {best_n}")
    return best_model, pd.DataFrame(resultados, columns=["States", "BIC", "AIC"])


def ajustar_hmm_sin_fuga(df, features=("Log_Return", "Volatility"),
                          train_ratio=TRAIN_RATIO, min_states=HMM_MIN_STATES,
                          max_states=HMM_MAX_STATES, cov_type=HMM_COV_TYPE):
    features = list(features)
    n = len(df)
    split_rows = int(train_ratio * n)
    scaler = StandardScaler()
    scaler.fit(df.loc[:split_rows - 1, features])
    X_all_scaled = scaler.transform(df[features])
    X_train_scaled = X_all_scaled[:split_rows]
    print(f"HMM: buscando estados ocultos (solo train, {split_rows} de {n} obs)")
    model, bic_df = seleccionar_n_estados_train(X_train_scaled, min_states, max_states, cov_type)
    filtered_states, filtered_probs = forward_filter(model, X_all_scaled)
    df_out = df.copy()
    df_out["State"] = filtered_states
    df_out["Split"] = ["train"] * split_rows + ["test"] * (n - split_rows)
    return df_out, model, scaler, split_rows, bic_df

## 11.5 Architecture of the Soft-Routing MoE Transformer

Components analogous to those of Section 5.

> **Development / supplementary implementation.**
> This block corresponds to an additional experimental implementation retained for methodological traceability.

In [ ]:
class PositionalEncoding(tf.keras.layers.Layer):
    def call(self, x):
        seq_len = tf.shape(x)[1]; d_dim = tf.shape(x)[2]
        positions = tf.cast(tf.range(seq_len)[:, None], tf.float32)
        dims = tf.cast(tf.range(d_dim)[None, :], tf.float32)
        angles = positions / tf.pow(10000.0, (2 * (dims // 2)) / tf.cast(d_dim, tf.float32))
        sines = tf.math.sin(angles[:, 0::2])
        cosines = tf.math.cos(angles[:, 1::2])
        pe = tf.reshape(tf.stack([sines, cosines], axis=2), [seq_len, -1])[:, :d_dim]
        return x + pe[None, :, :]


class MoEMergeLayer(tf.keras.layers.Layer):
    def __init__(self, n_experts, **kwargs):
        super().__init__(**kwargs); self.n_experts = n_experts
    def call(self, inputs):
        expert_outputs = inputs[:self.n_experts]
        routing_probs = inputs[-1]
        stacked = tf.stack(expert_outputs, axis=1)
        routed = tf.expand_dims(routing_probs, axis=-1)
        return tf.reduce_sum(stacked * routed, axis=1)


def transformer_block(x, num_heads, d_model, ff_dim, dropout_rate):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)(x, x)
    attn = Dropout(dropout_rate)(attn)
    x = LayerNormalization(epsilon=1e-6)(x + attn)
    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dropout(dropout_rate)(ffn)
    ffn = Dense(d_model)(ffn)
    ffn = Dropout(dropout_rate)(ffn)
    return LayerNormalization(epsilon=1e-6)(x + ffn)


def build_gater(seq_len, n_experts, n_features=1):
    inp = Input(shape=(seq_len, n_features))
    x = LSTM(64, activation="tanh")(inp)
    x = Dense(32, activation="relu")(x)
    return Model(inputs=inp, outputs=Dense(n_experts, activation="softmax")(x))


def make_expert_cfgs(n):
    cfgs = []
    for i in range(n):
        ratio = i / max(n - 1, 1)
        units = max(16, int(64 * (1 - 0.5 * ratio)))
        n_layers = 2 if ratio < 0.5 else 1
        drop = round(0.1 + 0.2 * ratio, 2)
        cfgs.append({"units": units, "n_layers": n_layers, "dropout": drop})
    return cfgs


def build_sparse_moe_transformer(seq_len, n_features, n_experts, expert_cfgs,
                                  num_heads, n_blocks, d_model, ff_dim, dropout):
    inp_seq = Input(shape=(seq_len, n_features), name="input_sequence")
    inp_routing = Input(shape=(n_experts,), name="input_routing_probs")
    x = Dense(d_model, name="input_projection")(inp_seq)
    x = PositionalEncoding(name="pos_enc")(x)
    x = Dropout(dropout)(x)
    for _ in range(n_blocks):
        x = transformer_block(x, num_heads, d_model, ff_dim, dropout)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    expert_outputs = []
    for i, cfg in enumerate(expert_cfgs):
        h = x
        for li in range(cfg["n_layers"]):
            h = Dense(cfg["units"], activation="relu", name=f"expert_{i}_h{li}")(h)
            if cfg.get("dropout", 0) > 0:
                h = Dropout(cfg["dropout"], name=f"expert_{i}_drop{li}")(h)
        expert_outputs.append(Dense(1, activation="linear", name=f"expert_{i}_out")(h))
    weighted_output = MoEMergeLayer(n_experts)(expert_outputs + [inp_routing])
    return Model(inputs=[inp_seq, inp_routing], outputs=weighted_output,
                 name="SparseMoE_Transformer")

## 11.6 Training function (Soft-Routing MoE, single seed)

Simplified version of the training function of Section 7: it trains only the Soft-Routing MoE Transformer and returns the metrics and the point-wise price predictions.

> **Development / supplementary implementation.**
> This block corresponds to an additional experimental implementation retained for methodological traceability.

In [ ]:
def entrenar_solo_sparse_moe(df, nombre, seed,
                                seq_length=SEQ_LENGTH, pred_len=1,
                                epochs=EPOCHS, batch_size=BATCH_SIZE,
                                trans_cfg=TRANS_CFG):
    set_seed(seed)
    print(f"\n[{nombre} | seed={seed}] Entrenando Sparse/Soft MoE Transformer...")

    df = df.copy()
    states = sorted(df["State"].unique())
    n_experts = len(states)
    state_to_expert = {s: i for i, s in enumerate(states)}
    expert_cfgs = make_expert_cfgs(n_experts)

    split_rows = int((df["Split"] == "train").sum())
    split_seq = split_rows - seq_length

    ret_train_raw = df["Log_Return"].values[:split_rows]
    ret_min, ret_max = ret_train_raw.min(), ret_train_raw.max()
    def norm(x): return (x - ret_min) / (ret_max - ret_min)
    def denorm(x): return x * (ret_max - ret_min) + ret_min

    df["Return_scaled"] = norm(df["Log_Return"].values)
    df["Volatility"] = df["Return_scaled"].rolling(window=5).std().fillna(0)
    vol_max = df["Volatility"].iloc[:split_rows].max()
    df["Volatility"] = df["Volatility"] / (vol_max if vol_max > 0 else 1)
    df["RSI"] = calculate_rsi(df["Price"], window=14).fillna(50) / 100.0

    feature_cols = ["Return_scaled", "Volatility", "RSI"]
    for state in states:
        col = f"HMM_{state}"
        df[col] = (df["State"] == state).astype(float)
        feature_cols.append(col)
    n_features = len(feature_cols)

    X_all, y_all = create_sequences_multivariate(
        df[feature_cols], df["Return_scaled"], seq_length, pred_len
    )
    X_train, X_test = X_all[:split_seq], X_all[split_seq:]
    y_train, y_test = y_all[:split_seq], y_all[split_seq:]

    n_val = int(len(X_train) * 0.15)
    X_tr, y_tr = X_train[:-n_val], y_train[:-n_val]
    X_val, y_val = X_train[-n_val:], y_train[-n_val:]

    # ---- Gater ----
    gater = build_gater(seq_length, n_experts, n_features)
    gater.compile(optimizer=Adam(1e-3), loss="categorical_crossentropy")
    train_states = df["State"].values[seq_length:split_rows]
    y_gater = np.zeros((len(train_states), n_experts))
    for i, s in enumerate(train_states):
        y_gater[i, state_to_expert[s]] = 1
    y_gater_tr, y_gater_val = y_gater[:-n_val], y_gater[-n_val:]

    gater.fit(X_tr, y_gater_tr, validation_data=(X_val, y_gater_val),
              epochs=50, batch_size=batch_size, verbose=0,
              callbacks=[EarlyStopping(patience=5, restore_best_weights=True,
                                        monitor="val_loss")])
    probs_tr   = gater.predict(X_tr,   verbose=0)
    probs_val  = gater.predict(X_val,  verbose=0)
    probs_test = gater.predict(X_test, verbose=0)

    # ---- Sparse MoE ----
    model_moe = build_sparse_moe_transformer(seq_length, n_features, n_experts,
                                              expert_cfgs, **trans_cfg)
    model_moe.compile(optimizer=Adam(5e-4), loss="mse")
    model_moe.fit([X_tr, probs_tr], y_tr,
                   validation_data=([X_val, probs_val], y_val),
                   epochs=epochs, batch_size=batch_size, verbose=0,
                   callbacks=[
                       EarlyStopping(monitor="val_loss", patience=15,
                                      restore_best_weights=True),
                       ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=5, min_lr=1e-6)
                   ])

    preds_moe = model_moe.predict([X_test, probs_test], verbose=0).flatten()

    true_ret     = denorm(y_test)
    pred_ret_moe = denorm(preds_moe)
    base_prices  = df["Price"].values[split_rows - 1:split_rows - 1 + len(pred_ret_moe)]
    prices_real  = base_prices * np.exp(true_ret)
    prices_moe   = base_prices * np.exp(pred_ret_moe)

    metrics = {
        "dataset":     nombre,
        "modelo":      "Sparse/Soft MoE Transformer",
        "seed":        seed,
        "MSE_price":   float(mean_squared_error(prices_real, prices_moe)),
        "MAE_price":   float(mean_absolute_error(prices_real, prices_moe)),
        "RMSE_price":  float(sqrt(mean_squared_error(prices_real, prices_moe))),
        "R2_price":    float(r2_score(prices_real, prices_moe)),
        "RMSE_return": float(sqrt(mean_squared_error(true_ret, pred_ret_moe))),
        "R2_return":   float(r2_score(true_ret, pred_ret_moe)),
    }

    tf.keras.backend.clear_session()

    return metrics, prices_real, prices_moe

## 11.7 Execution: HMM states from CSV and Soft-Routing MoE with the seven additional seeds

The HMM states are loaded from the CSV file uploaded in Step 4 (the same states used by the other models of the study) and the model is trained with the seven additional seeds. Results are saved to `resultados_variance_sparse/<DATASET>/`, so that they do not collide with other series.

**Accepted formats of the HMM-states CSV file**
- Minimum: `Date`, `State`
- Recommended: `Date`, `State`, `Split` (to respect the exact split of the pipeline)
- Complete: `Date`, `Price`, `State`, `Split`, ... (the relevant columns are used)

Estimated running time (7 seeds): approximately 30–40 min on CPU and 7–10 min on GPU.

In [ ]:
# ---- Paso 1: cargar dataset y estados HMM ya calculados ----
df_precio = cargar_precio(DATASET_CSV)
df_hmm_in = construir_features_hmm(df_precio)   # calcula Log_Return, Volatility

# Cargar estados HMM del CSV externo (los mismos que usaron los otros modelos del paper)
df_estados_ext = pd.read_csv(PATH_ESTADOS_HMM)
if "Date" in df_estados_ext.columns:
    df_estados_ext["Date"] = pd.to_datetime(df_estados_ext["Date"])

if "State" not in df_estados_ext.columns:
    raise ValueError(
        f"'{PATH_ESTADOS_HMM}' no contiene columna 'State'. Cols encontradas: {list(df_estados_ext.columns)}"
    )

# Estrategia de fusion segun columnas disponibles
if "Date" in df_estados_ext.columns:
    # Formato A/B/C con Date -> merge por Date
    cols_extra = ["Date", "State"]
    if "Split" in df_estados_ext.columns:
        cols_extra.append("Split")
    n_antes = len(df_hmm_in)
    df_estados = df_hmm_in.merge(df_estados_ext[cols_extra], on="Date", how="inner")
    n_despues = len(df_estados)
    if n_despues < n_antes:
        print(f"  Aviso: merge por Date descarto {n_antes - n_despues} filas "
              f"({n_antes} -> {n_despues}).")
elif len(df_estados_ext) == len(df_hmm_in):
    # Sin Date pero misma longitud -> asignar por indice
    df_estados = df_hmm_in.copy()
    df_estados["State"] = df_estados_ext["State"].values
    if "Split" in df_estados_ext.columns:
        df_estados["Split"] = df_estados_ext["Split"].values
    print("  Aviso: CSV sin columna 'Date'; asignados estados por indice.")
else:
    raise ValueError(
        f"El CSV de estados no tiene 'Date' y su longitud ({len(df_estados_ext)}) "
        f"no coincide con la del dataset ({len(df_hmm_in)})."
    )

# Si el CSV no trae Split, calcularlo con TRAIN_RATIO (mismo criterio que el pipeline)
if "Split" not in df_estados.columns:
    n = len(df_estados)
    split_rows = int(TRAIN_RATIO * n)
    df_estados["Split"] = ["train"] * split_rows + ["test"] * (n - split_rows)
    print(f"  Aviso: CSV sin columna 'Split'; calculado con TRAIN_RATIO={TRAIN_RATIO}.")
else:
    split_rows = int((df_estados["Split"] == "train").sum())

print(f"\nDataset: {DATASET_NOMBRE}  |  n_obs={len(df_estados)}  "
      f"|  train={split_rows}  |  test={len(df_estados)-split_rows}")
print(f"Estados HMM (cargados del CSV externo): {sorted(df_estados['State'].unique())}")
print(f"Distribucion de estados por split:")
print(df_estados.groupby(['Split', 'State']).size().unstack(fill_value=0))

# ---- Paso 2: entrenar Sparse MoE con las 7 semillas nuevas ----
registros_nuevos = []
predicciones_nuevas = {}  # seed -> {"prices_real": [...], "prices_pred": [...]}

for seed in SEEDS_NUEVAS:
    metrics, prices_real, prices_pred = entrenar_solo_sparse_moe(
        df_estados, DATASET_NOMBRE, seed=seed
    )
    registros_nuevos.append(metrics)
    predicciones_nuevas[str(seed)] = {
        "prices_real": prices_real.tolist(),
        "prices_pred": prices_pred.tolist(),
    }
    print(f"  MSE_price={metrics['MSE_price']:.4f}  "
          f"MAE_price={metrics['MAE_price']:.4f}  "
          f"R2_return={metrics['R2_return']:.4f}")

# ---- Guardar (en subcarpeta del dataset actual) ----
df_nuevos = pd.DataFrame(registros_nuevos)
df_nuevos.to_csv(os.path.join(DATASET_OUT_DIR, "sparse_moe_seeds_nuevas.csv"), index=False)

with open(os.path.join(DATASET_OUT_DIR, "sparse_moe_predicciones_nuevas.json"), "w") as f:
    json.dump({DATASET_NOMBRE: {"SparseMoE": predicciones_nuevas}}, f)

print("\n" + "="*70)
print(f"Resultados de las 7 semillas nuevas para {DATASET_NOMBRE}:")
print("="*70)
print(df_nuevos.to_string(index=False))

## 11.8 Combination with the results of the main pipeline (n = 10)

In [ ]:
# ---- Cargar las 3 semillas antiguas del pipeline original ----
df_antiguas = pd.read_csv(PATH_RES_ANTIGUAS)
mask = (df_antiguas["dataset"] == NOMBRE_EN_ANTIGUAS) & \
       (df_antiguas["modelo"]  == "Sparse/Soft MoE Transformer")
df_sparse_antiguas = df_antiguas[mask].copy()

if df_sparse_antiguas.empty:
    print(f"AVISO: no hay filas de '{NOMBRE_EN_ANTIGUAS}' con Sparse/Soft MoE Transformer "
          f"en {PATH_RES_ANTIGUAS}. Se seguirá solo con las 7 seeds nuevas.")
    df_todas = df_nuevos.copy()
else:
    print(f"Semillas antiguas encontradas para '{NOMBRE_EN_ANTIGUAS}': "
          f"{sorted(df_sparse_antiguas['seed'].tolist())}")
    cols_comunes = ["dataset", "modelo", "seed",
                    "MSE_price", "MAE_price", "RMSE_price", "R2_price",
                    "RMSE_return", "R2_return"]
    df_todas = pd.concat([
        df_sparse_antiguas[cols_comunes],
        df_nuevos[cols_comunes],
    ], ignore_index=True).sort_values("seed").reset_index(drop=True)

    # Normalizar nombre de dataset en la salida (por si en el CSV antiguo tenia otro alias)
    df_todas["dataset"] = DATASET_NOMBRE

df_todas.to_csv(os.path.join(DATASET_OUT_DIR, "sparse_moe_10_seeds.csv"), index=False)

print("\n" + "="*70)
print(f"Sparse MoE en {DATASET_NOMBRE} con {len(df_todas)} semillas:")
print("="*70)
print(df_todas[["seed", "MSE_price", "MAE_price", "RMSE_price", "R2_return"]].to_string(index=False))

# ---- Estadisticos descriptivos ----
print("\n" + "="*70)
print(f"{DATASET_NOMBRE} - Estadisticos descriptivos (comparativa n=3 vs n=10)")
print("="*70)

def resumen(v, label):
    v = np.asarray(v, dtype=float)
    return {
        "grupo":   label,
        "n":       len(v),
        "media":   v.mean(),
        "std":     v.std(ddof=1) if len(v) > 1 else 0.0,
        "min":     v.min(),
        "max":     v.max(),
        "mediana": np.median(v),
    }

filas_resumen = []
if not df_sparse_antiguas.empty:
    filas_resumen.append(resumen(df_sparse_antiguas["MSE_price"], "n=3 (paper)"))
filas_resumen.append(resumen(df_nuevos["MSE_price"],           "n=7 (nuevas)"))
filas_resumen.append(resumen(df_todas["MSE_price"],             f"n={len(df_todas)} (total)"))

resumen_df = pd.DataFrame(filas_resumen)
print(resumen_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## 11.9 Statistical tests against the reference baselines

- **Mann–Whitney U** (main test): Soft-Routing MoE (n = 10) versus each baseline (n = 3).
- **Wilcoxon signed-rank and paired t-test** (complementary): only on the three seeds shared by both groups.

In [ ]:
# ---- Cargar baselines modernos ----
df_baseline = pd.read_csv(PATH_RES_BASELINE)
df_baseline = df_baseline[df_baseline["seed"] != -1]  # excluye RandomWalk (determinista)
df_baseline_ds = df_baseline[df_baseline["dataset"] == NOMBRE_EN_BASELINE].copy()

if df_baseline_ds.empty:
    print(f"AVISO: no hay filas de '{NOMBRE_EN_BASELINE}' en {PATH_RES_BASELINE}.")
    print(f"Datasets disponibles en el CSV: {sorted(df_baseline['dataset'].unique())}")
    print("Vuelve a ejecutar la celda del uploader y elige el nombre correcto.")
else:
    baselines_disponibles = sorted(df_baseline_ds["modelo"].unique())
    print(f"Baselines modernos disponibles para '{NOMBRE_EN_BASELINE}': {baselines_disponibles}")

    METRICAS = ["MSE_price", "MAE_price", "RMSE_price", "RMSE_return"]
    ALPHA = 0.05

    rows = []
    for metrica in METRICAS:
        v_sparse_10 = df_todas[metrica].values.astype(float)                # n<=10

        for baseline in baselines_disponibles:
            b_sub = df_baseline_ds[df_baseline_ds["modelo"] == baseline].sort_values("seed")
            v_baseline = b_sub[metrica].values.astype(float)
            seeds_b = b_sub["seed"].tolist()

            # -- Test principal: Mann-Whitney U (no emparejado) --
            try:
                u_stat, p_mwu = mannwhitneyu(v_sparse_10, v_baseline, alternative="two-sided")
            except Exception:
                u_stat, p_mwu = np.nan, np.nan

            # -- Complementarios: tests emparejados sobre seeds comunes --
            if not df_sparse_antiguas.empty:
                v_sparse_paired = df_sparse_antiguas[
                    df_sparse_antiguas["seed"].isin(seeds_b)
                ].sort_values("seed")[metrica].values.astype(float)
            else:
                v_sparse_paired = np.array([])

            if len(v_sparse_paired) == len(v_baseline) and len(v_baseline) >= 2:
                try:
                    t_stat, p_t = ttest_rel(v_sparse_paired, v_baseline)
                except Exception:
                    t_stat, p_t = np.nan, np.nan
                try:
                    diff = v_sparse_paired - v_baseline
                    if np.all(diff == 0):
                        w_stat, p_w = np.nan, np.nan
                    else:
                        w_stat, p_w = wilcoxon(v_sparse_paired, v_baseline,
                                                zero_method="wilcox",
                                                alternative="two-sided",
                                                method="exact")
                except Exception:
                    w_stat, p_w = np.nan, np.nan
            else:
                t_stat, p_t, w_stat, p_w = np.nan, np.nan, np.nan, np.nan

            diff_media = v_sparse_10.mean() - v_baseline.mean()
            if np.isnan(p_mwu):
                veredicto = "no computable"
            elif p_mwu < ALPHA:
                veredicto = ("Sparse SIGNIFICATIVAMENTE PEOR" if diff_media > 0
                             else "Sparse SIGNIFICATIVAMENTE MEJOR")
            else:
                veredicto = "no significativo"

            rows.append({
                "dataset":        DATASET_NOMBRE,
                "metrica":        metrica,
                "baseline":       baseline,
                "n_sparse":       len(v_sparse_10),
                "sparse_mean":    float(v_sparse_10.mean()),
                "sparse_std":     float(v_sparse_10.std(ddof=1)) if len(v_sparse_10) > 1 else 0.0,
                "baseline_mean":  float(v_baseline.mean()),
                "baseline_std":   float(v_baseline.std(ddof=1)) if len(v_baseline) > 1 else 0.0,
                "diff_mean":      float(diff_media),
                "U_stat":         float(u_stat) if not np.isnan(u_stat) else np.nan,
                "p_MWU":          float(p_mwu)  if not np.isnan(p_mwu)  else np.nan,
                "t_stat_paired":  float(t_stat) if not np.isnan(t_stat) else np.nan,
                "p_t_paired":     float(p_t)    if not np.isnan(p_t)    else np.nan,
                "W_stat_paired":  float(w_stat) if not np.isnan(w_stat) else np.nan,
                "p_wilcoxon_paired": float(p_w) if not np.isnan(p_w)    else np.nan,
                "veredicto_MWU":  veredicto,
            })

    df_tests = pd.DataFrame(rows)
    df_tests.to_csv(os.path.join(DATASET_OUT_DIR, "tests_sparse_vs_baselines.csv"), index=False)

    for metrica in METRICAS:
        print("\n" + "="*95)
        print(f"{DATASET_NOMBRE} | {metrica}  -  Sparse MoE (n={len(df_todas)}) vs baseline (n=3)")
        print("="*95)
        sub = df_tests[df_tests["metrica"] == metrica]
        cols_show = ["baseline", "sparse_mean", "sparse_std",
                     "baseline_mean", "baseline_std", "diff_mean",
                     "p_MWU", "p_t_paired", "p_wilcoxon_paired",
                     "veredicto_MWU"]
        print(sub[cols_show].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    print(f"\nGuardado: {os.path.join(DATASET_OUT_DIR, 'tests_sparse_vs_baselines.csv')}")

## 11.10 Consolidation of all processed datasets

This cell detects all the datasets already processed (sub-folders of `resultados_variance_sparse/`) and consolidates the results into global tables. It is executed after the block has been run once for each series and generates:

- `sparse_moe_TODOS_seeds.csv`: metrics per dataset and seed (n = 10 per dataset).
- `sparse_moe_TODOS_agregado.csv`: mean, standard deviation and count per dataset.
- `tests_TODOS_datasets.csv`: Mann–Whitney U results for all comparisons.

In [ ]:
def consolidar_todos_los_datasets(out_dir=OUT_DIR):
    """Recorre resultados_variance_sparse/<DATASET>/ y consolida todo."""
    root = Path(out_dir)
    if not root.exists():
        print(f"No existe {root}. Ejecuta primero el notebook para al menos un dataset.")
        return

    subdirs = sorted([p for p in root.iterdir() if p.is_dir()])
    if not subdirs:
        print(f"{root} no contiene subcarpetas de dataset. Ejecuta primero el notebook.")
        return

    print(f"Datasets detectados: {[p.name for p in subdirs]}")

    # ---- Consolidar seeds ----
    dfs_seeds, dfs_tests = [], []
    for sub in subdirs:
        f10 = sub / "sparse_moe_10_seeds.csv"
        ft  = sub / "tests_sparse_vs_baselines.csv"
        if f10.exists():
            dfs_seeds.append(pd.read_csv(f10))
        if ft.exists():
            dfs_tests.append(pd.read_csv(ft))

    if not dfs_seeds:
        print("No se han encontrado CSVs 'sparse_moe_10_seeds.csv' en las subcarpetas.")
        return

    all_seeds = pd.concat(dfs_seeds, ignore_index=True)
    all_seeds.to_csv(root / "sparse_moe_TODOS_seeds.csv", index=False)

    # ---- Agregado por dataset ----
    METRICAS = ["MSE_price", "MAE_price", "RMSE_price", "R2_price",
                "RMSE_return", "R2_return"]
    metricas_ok = [m for m in METRICAS if m in all_seeds.columns]
    agg = all_seeds.groupby("dataset")[metricas_ok].agg(["mean", "std", "count"])
    agg.to_csv(root / "sparse_moe_TODOS_agregado.csv")

    print("\n" + "="*90)
    print("SPARSE MoE - AGREGADO POR DATASET (todas las semillas disponibles)")
    print("="*90)
    print(agg.round(4).to_string())

    # ---- Consolidar tests ----
    if dfs_tests:
        all_tests = pd.concat(dfs_tests, ignore_index=True)
        all_tests.to_csv(root / "tests_TODOS_datasets.csv", index=False)

        print("\n" + "="*90)
        print("TESTS MANN-WHITNEY U CONSOLIDADOS (Sparse vs cada baseline, por dataset y metrica)")
        print("="*90)

        for metrica in ["MSE_price", "MAE_price", "RMSE_return"]:
            sub = all_tests[all_tests["metrica"] == metrica]
            if sub.empty:
                continue
            print(f"\n--- {metrica} ---")
            cols = ["dataset", "baseline", "sparse_mean", "sparse_std",
                    "baseline_mean", "diff_mean", "p_MWU", "veredicto_MWU"]
            print(sub[cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    print(f"\nArchivos consolidados en: {root}/")
    for f in ["sparse_moe_TODOS_seeds.csv",
               "sparse_moe_TODOS_agregado.csv",
               "tests_TODOS_datasets.csv"]:
        p = root / f
        if p.exists():
            print(f"  {p}")


consolidar_todos_los_datasets()

## Usage notes for Section 11

### Suggested workflow

1. **First series:** upload the dataset CSV (e.g. `NVIDIA.csv`), `resultados_por_semilla.csv`, `metricas_por_semilla.csv` and the CSV with the HMM states of the dataset, and execute Sections 11.1 to 11.9.
2. **Subsequent series:** restart the runtime, execute the installation, imports/configuration and upload cells (11.1 and 11.2) and upload only the new dataset CSV and its HMM states: the two results files already on disk are detected automatically. Then execute the cells up to Section 11.9.
3. **Consolidation:** once every series has been processed, execute Section 11.10 to obtain the global tables.

### Generated file structure

```
resultados_variance_sparse/
├── NVIDIA/
│   ├── sparse_moe_seeds_nuevas.csv
│   ├── sparse_moe_10_seeds.csv
│   ├── sparse_moe_predicciones_nuevas.json
│   └── tests_sparse_vs_baselines.csv
├── APPLE/
│   └── ...
├── SP500/
│   └── ...
├── sparse_moe_TODOS_seeds.csv         (generated in Section 11.10)
├── sparse_moe_TODOS_agregado.csv      (generated in Section 11.10)
└── tests_TODOS_datasets.csv           (generated in Section 11.10)
```

### Choice of the statistical test

The **Mann–Whitney U** test is used as the main test instead of the Diebold–Mariano test because:

- it uses all ten seeds of the Soft-Routing MoE (the Diebold–Mariano test can only exploit the three seeds shared with the baselines);
- it does not require pairing: the reference baselines do not share seeds with the seven additional runs;
- it is non-parametric, and therefore robust to the asymmetric distribution of the MSE of the model;
- with n₁ = 10 and n₂ = 3 the minimum attainable p-value is approximately 0.007, so the null hypothesis can be rejected at conventional levels.

The point-wise predictions are stored in `sparse_moe_predicciones_nuevas.json`, so that a Diebold–Mariano test can additionally be run on the shared seeds if required.

# Appendix – Implementation Notes

This appendix gathers observations intended to help interpret the code. They document implementation details and are provided for traceability; the code is preserved without modification.

## A. Execution-specific configurations

- **Main pipeline (Section 1.3).** The cell has one active seed (`SEEDS = [42]`; the list `[42, 123, 2024]` appears commented) and one active dataset (`("SP500v3.csv", "SSP500")`; the APPLE and NVIDIA entries appear commented). `SSP500` is the dataset label used in the outputs of that execution. The grid-search options are `CACHE_GRID_SEARCH = True`, `SEED_GRID = 42` and `GRID_FAST = False`.
- **Synthetic generator (Section 2).** The configuration generates series of several sizes (`TAMANOS`); the seed of each series is `SEMILLA_BASE + n_samples`, advanced by one for each regeneration attempt. The four-state structure is the one configured in the cell.
- **Random Walk benchmark (Section 10).** The evaluated series are selected at execution time through an interactive upload.
- **Additional-seed analysis (Section 11).** The dataset is selected at execution time; its name is derived from the file name.

## B. Historical naming

- **"Sparse/Soft MoE".** Several identifiers and labels of the code (`build_sparse_moe_transformer`, `entrenar_sparse_moe`, the model name `SparseMoE_Transformer` and some messages) use this name. The label `"Soft-Routing MoE Transformer"` written in the metrics of Section 7 and the term used in the manuscript refer to the same model. Section 11 reads and writes the label `"Sparse/Soft MoE Transformer"`.
- **Language.** Comments, console messages, file names and some identifiers are in Spanish, as in the code used during development.

## C. Parameters that differ between implementation blocks

| Aspect | Hard-Routing HMM-MoE (Section 6) | Soft-Routing MoE (Sections 7 and 11) |
| --- | --- | --- |
| Number of epochs | `EPOCHS = 60` (Section 1.3) | `epochs=100` in `entrenar_sparse_moe`; `EPOCHS = 100` in Section 11 |
| Transformer feed-forward dimension | `ff_dim = 128` in the expert grid | `ff_dim = 256` in `TRANS_CFG` |
| Learning rate | 1e-3 (MLP, LSTM, BiLSTM) and 5e-4 (Transformer) | 5e-4 |
| Early stopping | experts: patience 5 on training loss; gater: patience 3 on training loss | patience 15 on validation loss (with `ReduceLROnPlateau`); gater: patience 5 on validation loss |
| Gater input | scaled log-returns only (`n_features = 1`) | multivariate features (returns, volatility, RSI and one-hot HMM states) |
| Scaling of returns | standardisation (training statistics) | min–max (training bounds) |

## D. Dependencies on external files and environment

- The main pipeline reads the price file declared in `DATASETS` and writes `<nombre>_HMM_States_corregido.csv`, `resultados_por_semilla.csv`, `resultados_agregados.csv` and `diebold_mariano_por_semilla.csv`.
- Section 11 reads `resultados_por_semilla.csv` (main pipeline), `metricas_por_semilla.csv` (per-seed metrics of the reference baselines, loaded from an external file) and a CSV with the HMM states of the dataset, whose file name can be adapted at upload time.
- Sections 10 and 11 rely on interactive upload (`google.colab.files`); Sections 1, 2 and 11 use `!pip install` and Section 2 uses `files.download`, so the notebook is designed for Google Colab.

## E. Supplementary analyses

- Section 11 (additional seeds, Mann–Whitney U, Wilcoxon and paired t-test) is a supplementary analysis and is not required to execute the main pipeline.
- The Diebold–Mariano test implemented in Section 9 compares, per seed, the global Transformer with the Soft-Routing MoE on the price scale.